In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  BINN Pseudo-Lineage Notebook — v2                                           ║
# ║  Biologically Informed Neural Network: Training + Inference                  ║
# ║                                                                              ║
# ║  REQUIRES: Run BINN_Preprocessing_v2 first to generate:                     ║
# ║    adata_preprocessed.h5ad, label_encoders.pkl, dims.json                   ║
# ║                                                                              ║
# ║  USAGE: Edit CONFIG section below, then run all cells end-to-end.           ║
# ║                                                                              ║
# ║  OUTPUT files (written to SAVE_DIR):                                         ║
# ║    binn_model.pt                  — trained model weights                   ║
# ║    binn_model_calibrated.pt       — model + temperature scaler              ║
# ║    adata_with_pseudolineages.h5ad — adata with pseudo_lineage assignments   ║
# ╚══════════════════════════════════════════════════════════════════════════════╝


# ══════════════════════════════════════════════════════════════════════════════
# CONFIG — edit this section only
# All dataset-specific and training variables live here.
# ══════════════════════════════════════════════════════════════════════════════

# ── Paths ─────────────────────────────────────────────────────────────────────
SAVE_DIR = "/srv/johan/Course work/Biologically Informed Neural Networks"
# Paths to preprocessing outputs — only change if you moved them
ADATA_PATH    = SAVE_DIR + "/adata_preprocessed.h5ad"
ENCODERS_PATH = SAVE_DIR + "/label_encoders.pkl"
DIMS_PATH     = SAVE_DIR + "/dims.json"

# ── Column names — must match what was used in preprocessing ─────────────────
CLONEID_COL   = "cloneid"          # Clone ID column in adata.obs
CELLTYPE_KEY  = "SCT_snn_res.2"    # Celltype column (adversarial anti-target)
UMAP_KEY      = "X_umap_neural_42" # UMAP key in adata.obsm for visualisation

# ── Training hyperparameters ──────────────────────────────────────────────────
N_EPOCHS       = 300    # Maximum training epochs
BATCH_SIZE     = 256    # Training batch size
LR             = 1e-3   # Learning rate
WEIGHT_DECAY   = 5e-4   # L2 regularisation
ADV_START      = 40     # Epoch at which adversarial gradient reversal begins
                        # Earlier = more disentanglement pressure throughout
PATIENCE       = 40     # Early stopping patience (epochs without bacc improvement)
MIN_EPOCHS     = 80     # Minimum epochs before early stopping is considered
                        # Must be > ADV_START to ensure adversarial phase is reached

# ── Loss weights ──────────────────────────────────────────────────────────────
LAMBDA_LINEAGE = 1.0    # Weight on primary lineage classification loss
LAMBDA_ADV     = 0.1    # Weight on adversarial celltype head (disentanglement)
LAMBDA_AUX     = 0.3    # Weight on per-modality auxiliary classification heads
                        # Higher = stronger pressure for each modality to learn independently

# ── Model architecture ────────────────────────────────────────────────────────
TOKEN_DIM      = 128    # Embedding dimension per modality token
Z_LINEAGE      = 64     # Lineage latent space dimension
Z_CELLTYPE     = 32     # Celltype latent space dimension

# ── Temperature calibration ───────────────────────────────────────────────────
CALIB_LR       = 0.01   # Learning rate for temperature scaler
CALIB_ITER     = 100    # LBFGS iterations for calibration

# ── Inference ─────────────────────────────────────────────────────────────────
INFER_BATCH    = 512    # Batch size during inference
CONF_THRESHOLD = 0.5    # Minimum confidence for a pseudo-lineage to be considered reliable

# ══════════════════════════════════════════════════════════════════════════════
# END CONFIG — do not edit below this line
# ══════════════════════════════════════════════════════════════════════════════


import json
import os
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Function
from torch.utils.data import Dataset, DataLoader, random_split
from scipy.sparse import issparse
from scipy.spatial.distance import jensenshannon
from sklearn.metrics import (
    balanced_accuracy_score, accuracy_score,
    confusion_matrix, classification_report,
    adjusted_rand_score, normalized_mutual_info_score,
)
from sklearn.preprocessing import LabelEncoder
from sklearn.dummy import DummyClassifier
import seaborn as sns
warnings.filterwarnings("ignore")

sc.settings.verbosity = 1

print("╔══════════════════════════════════════════════════════════════════╗")
print("║  BINN Pseudo-Lineage v2                                          ║")
print("╚══════════════════════════════════════════════════════════════════╝")
print()
print("── Configuration ────────────────────────────────────────────────────")
print(f"  SAVE_DIR       : {SAVE_DIR}")
print(f"  N_EPOCHS       : {N_EPOCHS}")
print(f"  BATCH_SIZE     : {BATCH_SIZE}")
print(f"  LR             : {LR}")
print(f"  ADV_START      : {ADV_START}")
print(f"  PATIENCE       : {PATIENCE}")
print(f"  MIN_EPOCHS     : {MIN_EPOCHS}")
print(f"  TOKEN_DIM      : {TOKEN_DIM}")
print(f"  Z_LINEAGE      : {Z_LINEAGE}")
print(f"  Z_CELLTYPE     : {Z_CELLTYPE}")
print("─────────────────────────────────────────────────────────────────────")

# ── Device ────────────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nDevice: {device}")
if torch.cuda.is_available():
    print(f"GPU   : {torch.cuda.get_device_name(0)}")


# ══════════════════════════════════════════════════════════════════════════════
# Step 1 — Load preprocessing outputs
# ══════════════════════════════════════════════════════════════════════════════
print("\n── Step 1: Load preprocessing outputs ──────────────────────────────")

with open(DIMS_PATH) as f:
    dims = json.load(f)
print(f"  dims.json    : {dims}")

adata = sc.read_h5ad(ADATA_PATH)
print(f"  adata        : {adata.n_obs:,} cells × {adata.n_vars:,} genes")

with open(ENCODERS_PATH, "rb") as f:
    encoders    = pickle.load(f)
le_lineage  = encoders["le_lineage"]
le_celltype = encoders["le_celltype"]
print(f"  Encoders     : le_lineage ({len(le_lineage.classes_)} classes), "
      f"le_celltype ({len(le_celltype.classes_)} classes)")

# ── Sanity check: dims.json must match adata exactly ─────────────────────────
lineage_check  = adata.obs.loc[adata.obs["lineage_code"] >= 0, "lineage_code"].nunique()
assert dims["n_lineages"] == lineage_check, (
    f"dims.json n_lineages={dims['n_lineages']} but adata has {lineage_check} — "
    f"rerun BINN_Preprocessing_v2 end-to-end"
)
assert dims["n_genes"] == adata.n_vars, (
    f"dims.json n_genes={dims['n_genes']} but adata has {adata.n_vars} — "
    f"rerun BINN_Preprocessing_v2 end-to-end"
)
assert dims["n_tfs"] == adata.obsm["X_tf"].shape[1], (
    f"dims.json n_tfs={dims['n_tfs']} mismatch — "
    f"rerun BINN_Preprocessing_v2 end-to-end"
)
print("  Sanity check : passed — dims.json matches adata")

# ── Clone size summary ────────────────────────────────────────────────────────
barcoded_mask    = adata.obs["lineage_code"] >= 0
barcoded_obs     = adata.obs.loc[barcoded_mask].copy()
cells_per_clone  = barcoded_obs.groupby(CLONEID_COL, observed=True).size()
cells_per_clone  = cells_per_clone[cells_per_clone > 0]

print(f"\n── Clone size summary ───────────────────────────────────────────────")
print(f"  min_clone_size   : {dims.get('min_clone_size', 'not saved')}")
print(f"  c2v_resolution   : {dims.get('c2v_resolution', 'not saved')}")
print(f"  Clones retained  : {len(cells_per_clone):,}")
print(f"  Clone size       : min={cells_per_clone.min()}  "
      f"max={cells_per_clone.max()}  mean={cells_per_clone.mean():.1f}")
print(f"  Barcoded cells   : {barcoded_mask.sum():,}")
print(f"  Lineage classes  : {dims['n_lineages']}")
print("─────────────────────────────────────────────────────────────────────")


# ══════════════════════════════════════════════════════════════════════════════
# Step 2 — Dataset
# ══════════════════════════════════════════════════════════════════════════════
print("\n── Step 2: Build dataset ────────────────────────────────────────────")

class BINNDataset(Dataset):
    """
    PyTorch Dataset wrapping all BINN modalities.
    barcoded_only=True  → only cells with lineage_code >= 0 (training)
    barcoded_only=False → all cells (inference)
    """
    def __init__(self, adata, barcoded_only=True):
        if barcoded_only:
            mask  = adata.obs["lineage_code"] >= 0
            adata = adata[mask].copy()

        X = adata.X.toarray() if issparse(adata.X) else np.array(adata.X)

        self.expr     = torch.tensor(X,                                  dtype=torch.float32)
        self.tf       = torch.tensor(adata.obsm["X_tf"],                 dtype=torch.float32)
        self.pathway  = torch.tensor(adata.obsm["X_pathway"],            dtype=torch.float32)
        self.go       = torch.tensor(adata.obsm["X_go"],                 dtype=torch.float32)
        self.lineage  = torch.tensor(adata.obs["lineage_code"].values,   dtype=torch.long)
        self.celltype = torch.tensor(adata.obs["celltype_code"].values,  dtype=torch.long)

        self.n_genes     = self.expr.shape[1]
        self.n_tfs       = self.tf.shape[1]
        self.n_pathways  = self.pathway.shape[1]
        self.n_go        = self.go.shape[1]
        self.n_lineages  = int(self.lineage.max().item()) + 1
        self.n_celltypes = int(self.celltype.max().item()) + 1

    def __len__(self):
        return self.expr.shape[0]

    def __getitem__(self, idx):
        return {
            "expr":     self.expr[idx],
            "tf":       self.tf[idx],
            "pathway":  self.pathway[idx],
            "go":       self.go[idx],
            "lineage":  self.lineage[idx],
            "celltype": self.celltype[idx],
            "cell_idx": idx,
        }

class BINNInferenceDataset(Dataset):
    """Dataset for inference — no labels required."""
    def __init__(self, adata):
        X = adata.X.toarray() if issparse(adata.X) else np.array(adata.X)
        self.expr      = torch.tensor(X,                       dtype=torch.float32)
        self.tf        = torch.tensor(adata.obsm["X_tf"],      dtype=torch.float32)
        self.pathway   = torch.tensor(adata.obsm["X_pathway"], dtype=torch.float32)
        self.go        = torch.tensor(adata.obsm["X_go"],      dtype=torch.float32)
        self.obs_names = adata.obs_names.tolist()

    def __len__(self):
        return self.expr.shape[0]

    def __getitem__(self, idx):
        return {
            "expr":     self.expr[idx],
            "tf":       self.tf[idx],
            "pathway":  self.pathway[idx],
            "go":       self.go[idx],
            "cell_idx": idx,
        }

dataset_full = BINNDataset(adata, barcoded_only=True)
n_val        = int(len(dataset_full) * dims.get("val_fraction", 0.15))
n_train      = len(dataset_full) - n_val

dataset_train, dataset_val = random_split(
    dataset_full, [n_train, n_val],
    generator=torch.Generator().manual_seed(dims.get("random_seed", 42)),
)

print(f"  Train    : {n_train:,} cells")
print(f"  Val      : {n_val:,} cells")
print(f"  Inference: {adata.n_obs:,} cells (all)")

# ── Class weights for imbalanced lineage classes ──────────────────────────────
lineage_counts = torch.bincount(dataset_full.lineage)
class_weights  = 1.0 / lineage_counts.float()
class_weights  = class_weights / class_weights.sum() * len(lineage_counts)
class_weights  = class_weights.to(device)
print(f"  Class weights computed ({len(lineage_counts)} classes)")


# ══════════════════════════════════════════════════════════════════════════════
# Step 3 — Model architecture
# ══════════════════════════════════════════════════════════════════════════════

# ── Gradient Reversal Layer ───────────────────────────────────────────────────
# Used for adversarial disentanglement: encoder is penalised for encoding
# celltype information in z_lineage
class GradientReversal(Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.clone()

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.alpha * grad_output, None

def grad_reverse(x, alpha=1.0):
    return GradientReversal.apply(x, alpha)

# ── Modality Encoder — MLP with LayerNorm + GELU ──────────────────────────────
class ModalityEncoder(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, out_dim),
            nn.LayerNorm(out_dim),
            nn.GELU(),
        )
    def forward(self, x):
        return self.net(x)

class ModalityEncoders(nn.Module):
    """
    Four independent encoders — one per modality.
    Higher dropout on larger modalities to prevent expression from dominating.
    """
    def __init__(self, dims, token_dim=TOKEN_DIM):
        super().__init__()
        self.enc_expr    = ModalityEncoder(dims["n_genes"],    512, token_dim, dropout=0.3)
        self.enc_tf      = ModalityEncoder(dims["n_tfs"],      128, token_dim, dropout=0.3)
        self.enc_pathway = ModalityEncoder(dims["n_pathways"],  32, token_dim, dropout=0.2)
        self.enc_go      = ModalityEncoder(dims["n_go"],       256, token_dim, dropout=0.3)

    def forward(self, expr, tf, pathway, go):
        return (
            self.enc_expr(expr),
            self.enc_tf(tf),
            self.enc_pathway(pathway),
            self.enc_go(go),
        )

# ── Cross-Modal Attention Fusion ──────────────────────────────────────────────
# Treats each modality embedding as a token and runs self-attention
# across the 4 modality tokens — allows modalities to contextualize each other
class CrossModalFusion(nn.Module):
    def __init__(self, token_dim=TOKEN_DIM, n_heads=4, dropout=0.1):
        super().__init__()
        self.attn  = nn.MultiheadAttention(
            embed_dim=token_dim, num_heads=n_heads,
            dropout=dropout, batch_first=True,
        )
        self.norm1 = nn.LayerNorm(token_dim)
        self.ff    = nn.Sequential(
            nn.Linear(token_dim, token_dim * 2),
            nn.GELU(),
            nn.Linear(token_dim * 2, token_dim),
        )
        self.norm2 = nn.LayerNorm(token_dim)
        # Store attention weights for interpretability
        self.last_attn_weights = None

    def forward(self, tokens):
        attn_out, attn_weights = self.attn(
            tokens, tokens, tokens,
            need_weights=True, average_attn_weights=True,
        )
        self.last_attn_weights = attn_weights.detach()
        tokens = self.norm1(tokens + attn_out)
        tokens = self.norm2(tokens + self.ff(tokens))
        return tokens.mean(dim=1)

# ── Latent Space — projects fused representation into z_lineage and z_celltype ─
class LatentSpace(nn.Module):
    def __init__(self, in_dim, z_lineage=Z_LINEAGE, z_celltype=Z_CELLTYPE):
        super().__init__()
        self.fc_lineage  = nn.Sequential(
            nn.Linear(in_dim, z_lineage),
            nn.LayerNorm(z_lineage),
            nn.GELU(),
        )
        self.fc_celltype = nn.Sequential(
            nn.Linear(in_dim, z_celltype),
            nn.LayerNorm(z_celltype),
            nn.GELU(),
        )

    def forward(self, x):
        return self.fc_lineage(x), self.fc_celltype(x)

# ── Output Heads ──────────────────────────────────────────────────────────────
class LineageHead(nn.Module):
    """Primary head — predicts clonal lineage cluster from z_lineage."""
    def __init__(self, z_dim, n_lineages):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(z_dim, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(64, n_lineages),
        )
    def forward(self, z):
        return self.net(z)

class CelltypeHead(nn.Module):
    """Adversarial head — predicts celltype from z_lineage via gradient reversal.
    Encoder is trained to FOOL this head, ensuring z_lineage does not encode
    cell type information."""
    def __init__(self, z_dim, n_celltypes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(z_dim, 32),
            nn.GELU(),
            nn.Linear(32, n_celltypes),
        )
    def forward(self, z):
        return self.net(z)

class ModalityAuxHead(nn.Module):
    """Per-modality auxiliary head — forces each encoder to independently learn
    lineage-relevant features, preventing modality collapse where the model
    ignores all but the strongest modality (expression)."""
    def __init__(self, token_dim, n_lineages):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(token_dim, 32),
            nn.GELU(),
            nn.Linear(32, n_lineages),
        )
    def forward(self, z):
        return self.net(z)

# ── Full BINN Model ───────────────────────────────────────────────────────────
class BINN(nn.Module):
    def __init__(self, dims, token_dim=TOKEN_DIM,
                 z_lineage=Z_LINEAGE, z_celltype=Z_CELLTYPE):
        super().__init__()
        self.encoders      = ModalityEncoders(dims, token_dim=token_dim)
        self.fusion        = CrossModalFusion(token_dim=token_dim, n_heads=4)
        self.latent        = LatentSpace(in_dim=token_dim,
                                         z_lineage=z_lineage,
                                         z_celltype=z_celltype)
        self.lineage_head  = LineageHead(z_lineage, dims["n_lineages"])
        self.celltype_head = CelltypeHead(z_lineage, dims["n_celltypes"])
        # Auxiliary heads — one per modality encoder
        self.aux_expr    = ModalityAuxHead(token_dim, dims["n_lineages"])
        self.aux_tf      = ModalityAuxHead(token_dim, dims["n_lineages"])
        self.aux_pathway = ModalityAuxHead(token_dim, dims["n_lineages"])
        self.aux_go      = ModalityAuxHead(token_dim, dims["n_lineages"])

    def forward(self, batch, alpha=0.0):
        # Encode each modality independently
        z_e, z_t, z_p, z_g = self.encoders(
            batch["expr"], batch["tf"], batch["pathway"], batch["go"]
        )
        # Per-modality auxiliary predictions (train time only)
        aux_logits = {
            "expr":    self.aux_expr(z_e),
            "tf":      self.aux_tf(z_t),
            "pathway": self.aux_pathway(z_p),
            "go":      self.aux_go(z_g),
        }
        # Cross-modal attention fusion
        tokens  = torch.stack([z_e, z_t, z_p, z_g], dim=1)
        z_fused = self.fusion(tokens)
        # Decompose into lineage and celltype subspaces
        z_lin, z_ct = self.latent(z_fused)
        # Primary lineage prediction
        logits_lineage  = self.lineage_head(z_lin)
        # Adversarial celltype prediction (gradient reversal applied)
        z_rev           = grad_reverse(z_lin, alpha=alpha)
        logits_celltype = self.celltype_head(z_rev)

        return {
            "logits_lineage":  logits_lineage,
            "logits_celltype": logits_celltype,
            "aux_logits":      aux_logits,
            "z_lineage":       z_lin,
            "z_celltype":      z_ct,
            "attn_weights":    self.fusion.last_attn_weights,
        }

# ── Loss Function ─────────────────────────────────────────────────────────────
class BINNLoss(nn.Module):
    def __init__(self, lambda_lineage=LAMBDA_LINEAGE, lambda_adv=LAMBDA_ADV,
                 lambda_aux=LAMBDA_AUX, class_weights=None):
        super().__init__()
        self.lambda_lineage = lambda_lineage
        self.lambda_adv     = lambda_adv
        self.lambda_aux     = lambda_aux
        self.ce             = nn.CrossEntropyLoss(weight=class_weights)
        self.ce_unweighted  = nn.CrossEntropyLoss()

    def forward(self, out, batch):
        l_lineage = self.ce(out["logits_lineage"],             batch["lineage"])
        l_adv     = self.ce_unweighted(out["logits_celltype"], batch["celltype"])
        l_aux     = sum(
            self.ce(logits, batch["lineage"])
            for logits in out["aux_logits"].values()
        ) / len(out["aux_logits"])

        total = (self.lambda_lineage * l_lineage +
                 self.lambda_adv     * l_adv     +
                 self.lambda_aux     * l_aux)

        return total, {
            "lineage":     l_lineage.item(),
            "adv":         l_adv.item(),
            "aux":         l_aux.item(),
            "aux_expr":    self.ce(out["aux_logits"]["expr"],    batch["lineage"]).item(),
            "aux_tf":      self.ce(out["aux_logits"]["tf"],      batch["lineage"]).item(),
            "aux_pathway": self.ce(out["aux_logits"]["pathway"], batch["lineage"]).item(),
            "aux_go":      self.ce(out["aux_logits"]["go"],      batch["lineage"]).item(),
        }

# Instantiate model
model    = BINN(dims).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n── Model ────────────────────────────────────────────────────────────")
print(f"  Trainable parameters: {n_params:,}")


# ══════════════════════════════════════════════════════════════════════════════
# Step 4 — Training
# ══════════════════════════════════════════════════════════════════════════════
print("\n── Step 4: Training ─────────────────────────────────────────────────")

def train_binn(
    model, dataset_train, dataset_val, dims,
    n_epochs=N_EPOCHS, batch_size=BATCH_SIZE, lr=LR,
    adv_start_epoch=ADV_START, patience=PATIENCE, min_epochs=MIN_EPOCHS,
    class_weights=None, device=device,
):
    loader_train = DataLoader(dataset_train, batch_size=batch_size,
                              shuffle=True,  num_workers=0)
    loader_val   = DataLoader(dataset_val,   batch_size=batch_size,
                              shuffle=False, num_workers=0)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr,
                                  weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                    optimizer, T_max=n_epochs)
    criterion = BINNLoss(class_weights=class_weights)

    history       = {"train": [], "val": []}
    best_val_bacc = 0.0
    best_state    = None
    epochs_no_imp = 0

    for epoch in range(1, n_epochs + 1):

        # Alpha ramp: 0 → 1 smoothly after adv_start_epoch
        if epoch < adv_start_epoch:
            alpha = 0.0
        else:
            progress = (epoch - adv_start_epoch) / (n_epochs - adv_start_epoch)
            alpha    = float(2.0 / (1.0 + np.exp(-10 * progress)) - 1)

        # ── Train ─────────────────────────────────────────────────────────────
        model.train()
        train_losses = []
        for batch in loader_train:
            batch = {k: v.to(device) for k, v in batch.items()}
            optimizer.zero_grad()
            out  = model(batch, alpha=alpha)
            loss, loss_dict = criterion(out, batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_losses.append(loss_dict)

        # ── Validate ──────────────────────────────────────────────────────────
        model.eval()
        val_losses            = []
        correct, total        = 0, 0
        all_preds, all_labels = [], []
        with torch.no_grad():
            for batch in loader_val:
                batch = {k: v.to(device) for k, v in batch.items()}
                out   = model(batch, alpha=0.0)
                _, loss_dict = criterion(out, batch)
                val_losses.append(loss_dict)
                preds    = out["logits_lineage"].argmax(dim=1)
                correct += (preds == batch["lineage"]).sum().item()
                total   += len(batch["lineage"])
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(batch["lineage"].cpu().numpy())

        scheduler.step()

        def mean_dict(dl):
            keys = dl[0].keys()
            return {k: np.mean([d[k] for d in dl]) for k in keys}

        t    = mean_dict(train_losses)
        v    = mean_dict(val_losses)
        acc  = correct / total * 100
        bacc = balanced_accuracy_score(all_labels, all_preds) * 100

        history["train"].append(t)
        history["val"].append(v)

        if bacc > best_val_bacc:
            best_val_bacc = bacc
            best_state    = {k: val.cpu().clone()
                             for k, val in model.state_dict().items()}
            epochs_no_imp = 0
            star          = "★"
        else:
            epochs_no_imp += 1
            star          = ""

        if epoch % 10 == 0 or epoch == 1:
            print(f"Epoch {epoch:3d} α={alpha:.2f} | "
                  f"lin={t['lineage']:.3f} adv={t['adv']:.3f} "
                  f"aux={t['aux']:.3f} "
                  f"[e={t['aux_expr']:.2f} tf={t['aux_tf']:.2f} "
                  f"pw={t['aux_pathway']:.2f} go={t['aux_go']:.2f}] | "
                  f"val_lin={v['lineage']:.3f} | "
                  f"acc={acc:.1f}% bacc={bacc:.1f}% {star}")

        if epoch >= min_epochs and epochs_no_imp >= patience:
            print(f"\nEarly stopping at epoch {epoch} — "
                  f"best bacc={best_val_bacc:.1f}%")
            break

    model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
    print(f"\nRestored best model — bacc={best_val_bacc:.1f}%")
    return model, history


model, history = train_binn(
    model, dataset_train, dataset_val, dims,
    class_weights=class_weights,
    device=device,
)

# ── Save initial model ────────────────────────────────────────────────────────
os.makedirs(SAVE_DIR, exist_ok=True)
torch.save({
    "model_state_dict": model.state_dict(),
    "dims":             dims,
    "history":          history,
}, os.path.join(SAVE_DIR, "binn_model.pt"))
print(f"  Saved: binn_model.pt")


# ══════════════════════════════════════════════════════════════════════════════
# Step 5 — Null model validation
# Establishes baselines to verify BINN is learning above chance
# ══════════════════════════════════════════════════════════════════════════════
print("\n── Step 5: Null model validation ────────────────────────────────────")

val_indices   = dataset_val.indices
train_indices = dataset_train.indices
val_lineage   = dataset_full.lineage[val_indices].numpy()
train_lineage = dataset_full.lineage[train_indices].numpy()
val_expr      = dataset_full.expr[val_indices].numpy()
train_expr    = dataset_full.expr[train_indices].numpy()

# Null 1: Random
rnd       = DummyClassifier(strategy="uniform", random_state=42)
rnd.fit(train_expr, train_lineage)
rnd_bacc  = balanced_accuracy_score(val_lineage, rnd.predict(val_expr)) * 100

# Null 2: Majority class
maj       = DummyClassifier(strategy="most_frequent", random_state=42)
maj.fit(train_expr, train_lineage)
maj_bacc  = balanced_accuracy_score(val_lineage, maj.predict(val_expr)) * 100

# Null 3: Transcriptomics-only MLP
class TranscriptomicsOnlyMLP(nn.Module):
    def __init__(self, n_genes, n_lineages):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_genes, 256), nn.LayerNorm(256), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(256, 128), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(128, n_lineages),
        )
    def forward(self, x):
        return self.net(x)

null_model    = TranscriptomicsOnlyMLP(dims["n_genes"], dims["n_lineages"]).to(device)
null_optim    = torch.optim.Adam(null_model.parameters(), lr=LR,
                                  weight_decay=WEIGHT_DECAY)
null_criterion= nn.CrossEntropyLoss(weight=class_weights)
loader_null   = DataLoader(dataset_train, batch_size=BATCH_SIZE, shuffle=True)
loader_val_n  = DataLoader(dataset_val,   batch_size=BATCH_SIZE, shuffle=False)

best_null_bacc  = 0.0
best_null_state = None
print("  Training transcriptomics-only null model...")
for epoch in range(1, N_EPOCHS + 1):
    null_model.train()
    for batch in loader_null:
        batch = {k: v.to(device) for k, v in batch.items()}
        null_optim.zero_grad()
        loss = null_criterion(null_model(batch["expr"]), batch["lineage"])
        loss.backward()
        null_optim.step()
    if epoch % 50 == 0:
        null_model.eval()
        preds_n, labels_n = [], []
        with torch.no_grad():
            for batch in loader_val_n:
                batch = {k: v.to(device) for k, v in batch.items()}
                preds_n.extend(null_model(batch["expr"]).argmax(dim=1).cpu().numpy())
                labels_n.extend(batch["lineage"].cpu().numpy())
        nb = balanced_accuracy_score(labels_n, preds_n) * 100
        if nb > best_null_bacc:
            best_null_bacc  = nb
            best_null_state = {k: v.cpu().clone()
                               for k, v in null_model.state_dict().items()}
        print(f"    Epoch {epoch:3d} | bacc={nb:.1f}%")

null_model.load_state_dict({k: v.to(device) for k, v in best_null_state.items()})

# BINN val bacc
model.eval()
binn_preds, binn_labels = [], []
with torch.no_grad():
    for batch in DataLoader(dataset_val, batch_size=BATCH_SIZE, shuffle=False):
        batch = {k: v.to(device) for k, v in batch.items()}
        preds = model(batch, alpha=0.0)["logits_lineage"].argmax(dim=1)
        binn_preds.extend(preds.cpu().numpy())
        binn_labels.extend(batch["lineage"].cpu().numpy())
binn_bacc = balanced_accuracy_score(binn_labels, binn_preds) * 100

print(f"\n── Null model summary ───────────────────────────────────────────────")
print(f"  Random chance (1/{dims['n_lineages']} classes)  : "
      f"{100/dims['n_lineages']:.1f}%")
print(f"  Null 1 — random baseline       : bacc={rnd_bacc:.1f}%")
print(f"  Null 2 — majority class        : bacc={maj_bacc:.1f}%")
print(f"  Null 3 — transcriptomics MLP   : bacc={best_null_bacc:.1f}%")
print(f"  BINN (multi-modal)             : bacc={binn_bacc:.1f}%")
print(f"  BINN improvement over null MLP : {binn_bacc - best_null_bacc:+.1f}%")
print("─────────────────────────────────────────────────────────────────────")


# ══════════════════════════════════════════════════════════════════════════════
# Step 6 — Ablation study
# Quantifies contribution of each modality by zeroing it out at inference
# ══════════════════════════════════════════════════════════════════════════════
print("\n── Step 6: Ablation study ───────────────────────────────────────────")

def eval_ablation(model, dataset_val, device, ablate=None):
    loader = DataLoader(dataset_val, batch_size=BATCH_SIZE, shuffle=False)
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            if ablate == "tf":
                batch["tf"]      = torch.zeros_like(batch["tf"])
            elif ablate == "pathway":
                batch["pathway"] = torch.zeros_like(batch["pathway"])
            elif ablate == "go":
                batch["go"]      = torch.zeros_like(batch["go"])
            elif ablate == "expr":
                batch["expr"]    = torch.zeros_like(batch["expr"])
            preds = model(batch, alpha=0.0)["logits_lineage"].argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(batch["lineage"].cpu().numpy())
    return balanced_accuracy_score(all_labels, all_preds) * 100

print(f"  Full model (all modalities) : bacc={binn_bacc:.1f}%")
for ablate in ["expr", "tf", "pathway", "go"]:
    bacc = eval_ablation(model, dataset_val, device, ablate=ablate)
    drop = binn_bacc - bacc
    print(f"  Without {ablate:10s}         : bacc={bacc:.1f}%  (drop={drop:+.1f}%)")

# ── Attention weights ─────────────────────────────────────────────────────────
model.eval()
all_attn = []
with torch.no_grad():
    for batch in DataLoader(dataset_val, batch_size=BATCH_SIZE, shuffle=False):
        batch = {k: v.to(device) for k, v in batch.items()}
        model(batch, alpha=0.0)
        all_attn.append(model.fusion.last_attn_weights.cpu())

attn_mean  = torch.cat(all_attn, dim=0).mean(dim=0)
modalities = ["expr", "tf", "pathway", "go"]
print(f"\n  Mean attention weights (rows=query, cols=key):")
print(f"  {'':10s}  " + "  ".join(f"{m:>8s}" for m in modalities))
for i, row_name in enumerate(modalities):
    row = "  ".join(f"{attn_mean[i, j].item():8.3f}" for j in range(4))
    print(f"  {row_name:10s}  {row}")


# ══════════════════════════════════════════════════════════════════════════════
# Step 7 — Temperature calibration
# Rescales softmax probabilities so confidence is meaningful
# Without calibration: high confidence ≠ high accuracy
# ══════════════════════════════════════════════════════════════════════════════
print("\n── Step 7: Temperature calibration ─────────────────────────────────")

class TemperatureScaler(nn.Module):
    def __init__(self):
        super().__init__()
        self.temperature = nn.Parameter(torch.ones(1))

    def forward(self, logits):
        return logits / self.temperature

    def calibrate(self, model, dataset_val, device,
                  lr=CALIB_LR, n_iter=CALIB_ITER):
        loader = DataLoader(dataset_val, batch_size=BATCH_SIZE, shuffle=False)
        model.eval()
        all_logits, all_labels = [], []
        with torch.no_grad():
            for batch in loader:
                batch = {k: v.to(device) for k, v in batch.items()}
                out   = model(batch, alpha=0.0)
                all_logits.append(out["logits_lineage"].cpu())
                all_labels.append(batch["lineage"].cpu())

        all_logits = torch.cat(all_logits)
        all_labels = torch.cat(all_labels)

        optimizer = torch.optim.LBFGS([self.temperature], lr=lr, max_iter=n_iter)
        nll       = nn.CrossEntropyLoss()

        def eval_step():
            optimizer.zero_grad()
            loss = nll(all_logits / self.temperature, all_labels)
            loss.backward()
            return loss

        optimizer.step(eval_step)
        print(f"  Optimal temperature: {self.temperature.item():.4f}")
        return self

scaler = TemperatureScaler().to(device)
scaler.calibrate(model, dataset_val, device)

# ── Evaluate calibrated model ─────────────────────────────────────────────────
model.eval()
all_preds_cal, all_labels_cal, all_probs_cal = [], [], []
with torch.no_grad():
    for batch in DataLoader(dataset_val, batch_size=BATCH_SIZE, shuffle=False):
        batch      = {k: v.to(device) for k, v in batch.items()}
        out        = model(batch, alpha=0.0)
        logits_cal = scaler(out["logits_lineage"])
        probs_cal  = torch.softmax(logits_cal, dim=-1)
        preds_cal  = probs_cal.argmax(dim=-1)
        all_preds_cal.extend(preds_cal.cpu().numpy())
        all_labels_cal.extend(batch["lineage"].cpu().numpy())
        all_probs_cal.extend(probs_cal.cpu().numpy())

all_preds_cal  = np.array(all_preds_cal)
all_labels_cal = np.array(all_labels_cal)
all_probs_cal  = np.array(all_probs_cal)
confidence_cal = all_probs_cal.max(axis=1)
bacc_cal       = balanced_accuracy_score(all_labels_cal, all_preds_cal) * 100
acc_cal        = accuracy_score(all_labels_cal, all_preds_cal) * 100

print(f"\n  After calibration:")
print(f"    Accuracy          : {acc_cal:.2f}%")
print(f"    Balanced accuracy : {bacc_cal:.2f}%")
print(f"    Mean confidence   : {confidence_cal.mean():.3f}")
print(f"    Conf > 0.9        : {(confidence_cal > 0.9).sum()} / {len(confidence_cal)} "
      f"({(confidence_cal > 0.9).mean()*100:.1f}%)")

# ── Calibration curve ─────────────────────────────────────────────────────────
correct_cal = (all_preds_cal == all_labels_cal)
bins        = np.arange(0, 1.05, 0.1)
print(f"\n── Calibrated confidence vs accuracy ───────────────────────────────")
print(f"  {'Conf range':15s}  {'Accuracy':>10s}  {'N cells':>10s}")
for lo, hi in zip(bins[:-1], bins[1:]):
    mask = (confidence_cal >= lo) & (confidence_cal < hi)
    if mask.sum() > 0:
        print(f"  {lo:.1f} – {hi:.1f}          "
              f"  {correct_cal[mask].mean()*100:9.1f}%  {mask.sum():10d}")


# ══════════════════════════════════════════════════════════════════════════════
# Step 8 — Inference on all cells
# ══════════════════════════════════════════════════════════════════════════════
print("\n── Step 8: Inference ────────────────────────────────────────────────")

infer_dataset = BINNInferenceDataset(adata)
infer_loader  = DataLoader(infer_dataset, batch_size=INFER_BATCH, shuffle=False)

model.eval()
all_z, all_probs_inf, all_preds_inf = [], [], []
with torch.no_grad():
    for batch in infer_loader:
        batch_gpu  = {k: v.to(device) for k, v in batch.items()
                      if k != "cell_idx"}
        out        = model(batch_gpu, alpha=0.0)
        logits_cal = scaler(out["logits_lineage"])
        probs      = torch.softmax(logits_cal, dim=-1)
        preds      = probs.argmax(dim=-1)
        all_z.append(out["z_lineage"].cpu().numpy())
        all_probs_inf.append(probs.cpu().numpy())
        all_preds_inf.append(preds.cpu().numpy())

z_lineage   = np.concatenate(all_z,          axis=0)
probs_inf   = np.concatenate(all_probs_inf,   axis=0)
pred_labels = np.concatenate(all_preds_inf,   axis=0)
confidence  = probs_inf.max(axis=1)

adata.obs["pseudo_lineage"]      = le_lineage.inverse_transform(pred_labels)
adata.obs["pseudo_lineage_conf"] = confidence
adata.obs["is_barcoded"]         = barcoded_mask
adata.obsm["X_lineage"]          = z_lineage

unbarcoded_mask = ~barcoded_mask
high_conf_mask  = adata.obs["pseudo_lineage_conf"] > CONF_THRESHOLD

print(f"  Inference complete: {len(pred_labels):,} cells")
print(f"  Barcoded   (n={barcoded_mask.sum():5,}): "
      f"mean conf={adata.obs.loc[barcoded_mask,   'pseudo_lineage_conf'].mean():.3f}  "
      f"median={adata.obs.loc[barcoded_mask,   'pseudo_lineage_conf'].median():.3f}")
print(f"  Unbarcoded (n={unbarcoded_mask.sum():5,}): "
      f"mean conf={adata.obs.loc[unbarcoded_mask,  'pseudo_lineage_conf'].mean():.3f}  "
      f"median={adata.obs.loc[unbarcoded_mask,  'pseudo_lineage_conf'].median():.3f}")
print(f"  Conf > {CONF_THRESHOLD}: {high_conf_mask.sum():,} / {len(adata):,} cells "
      f"({high_conf_mask.mean()*100:.1f}%)")
print(f"  Of which unbarcoded: {(high_conf_mask & unbarcoded_mask).sum():,}")


# ══════════════════════════════════════════════════════════════════════════════
# Step 9 — Validation metrics
# ══════════════════════════════════════════════════════════════════════════════
print("\n── Step 9: Validation metrics ───────────────────────────────────────")

# Per-class accuracy
per_class_acc = []
for c in range(dims["n_lineages"]):
    mask = all_labels_cal == c
    if mask.sum() == 0:
        per_class_acc.append(np.nan)
    else:
        per_class_acc.append((all_preds_cal[mask] == c).mean() * 100)

print(f"  Accuracy          : {acc_cal:.2f}%")
print(f"  Balanced accuracy : {bacc_cal:.2f}%")
print(f"  Random baseline   : {100/dims['n_lineages']:.2f}%")
print(f"  Null MLP          : {best_null_bacc:.2f}%")
print(f"  BINN improvement  : {bacc_cal - best_null_bacc:+.2f}%")
print(f"\n  Per-class accuracy:")
for c, a in enumerate(per_class_acc):
    n   = (all_labels_cal == c).sum()
    bar = "█" * int(a / 5) if not np.isnan(a) else ""
    print(f"    Class {c:2d} (n={n:4d}) : {a:5.1f}%  {bar}")

# Confusion matrix plot
cm      = confusion_matrix(all_labels_cal, all_preds_cal)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
class_names = [str(i) for i in range(dims["n_lineages"])]

fig, axes = plt.subplots(1, 2, figsize=(20, 8))
sns.heatmap(cm_norm, ax=axes[0], cmap="Blues", vmin=0, vmax=1,
            xticklabels=class_names, yticklabels=class_names,
            linewidths=0.3, linecolor="white")
axes[0].set_title("Normalised confusion matrix")
axes[0].set_xlabel("Predicted lineage")
axes[0].set_ylabel("True lineage")

valid_classes = [c for c in range(dims["n_lineages"])
                 if not np.isnan(per_class_acc[c])]
valid_accs    = [per_class_acc[c] for c in valid_classes]
valid_counts  = [(all_labels_cal == c).sum() for c in valid_classes]
colors        = ["#1D9E75" if a >= 70 else "#EF9F27" if a >= 50
                 else "#E24B4A" for a in valid_accs]
axes[1].barh([f"Class {c} (n={valid_counts[i]})" for i, c in enumerate(valid_classes)],
             valid_accs, color=colors)
axes[1].axvline(x=bacc_cal, color="black", linestyle="--",
                linewidth=1, label=f"BINN bacc={bacc_cal:.1f}%")
axes[1].axvline(x=best_null_bacc, color="gray", linestyle=":",
                linewidth=1, label=f"Null MLP={best_null_bacc:.1f}%")
axes[1].set_xlabel("Per-class accuracy (%)")
axes[1].set_title("Per-class accuracy")
axes[1].legend(fontsize=9)
axes[1].set_xlim(0, 100)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "validation_metrics.png"),
            dpi=150, bbox_inches="tight")
plt.show()
print(f"  Saved: validation_metrics.png")


# ══════════════════════════════════════════════════════════════════════════════
# Step 10 — Compare pseudo-lineages to clone2vec ground truth
# ══════════════════════════════════════════════════════════════════════════════
print("\n── Step 10: Clone2vec comparison ────────────────────────────────────")

barcoded   = adata[barcoded_mask].copy()
barcoded.obs["true_lineage"] = le_lineage.inverse_transform(
    barcoded.obs["lineage_code"].values
)

ari  = adjusted_rand_score(barcoded.obs["true_lineage"],
                            barcoded.obs["pseudo_lineage"])
nmi  = normalized_mutual_info_score(barcoded.obs["true_lineage"],
                                     barcoded.obs["pseudo_lineage"])
agreement = (barcoded.obs["true_lineage"] == barcoded.obs["pseudo_lineage"]).mean()

print(f"  Accuracy (exact match)   : {agreement*100:.1f}%")
print(f"  Adjusted Rand Index      : {ari:.4f}")
print(f"  Normalized Mutual Info   : {nmi:.4f}")

# Distribution comparison
unbarcoded   = adata[unbarcoded_mask].copy()
bc_dist      = barcoded.obs["pseudo_lineage"].value_counts(normalize=True).sort_index()
unbc_dist    = unbarcoded.obs["pseudo_lineage"].value_counts(normalize=True).sort_index()
dist_df      = pd.DataFrame({"barcoded": bc_dist, "unbarcoded": unbc_dist}).fillna(0)
jsd          = jensenshannon(dist_df["barcoded"], dist_df["unbarcoded"])

print(f"\n  Jensen-Shannon divergence (barcoded vs unbarcoded): {jsd:.4f}")
print(f"  (0.0 = identical distributions, 1.0 = completely different)")


# ══════════════════════════════════════════════════════════════════════════════
# Step 11 — Visualisation
# ══════════════════════════════════════════════════════════════════════════════
print("\n── Step 11: Visualisation ───────────────────────────────────────────")

# ── UMAP overview ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(22, 7))
sc.pl.embedding(adata, basis=UMAP_KEY, color="pseudo_lineage",
                title=f"Pseudo-lineage (all {adata.n_obs:,} cells)",
                frameon=False, legend_loc="on data", legend_fontsize=5,
                ax=axes[0], show=False)
sc.pl.embedding(adata, basis=UMAP_KEY, color="pseudo_lineage_conf",
                title="Calibrated confidence", frameon=False,
                cmap="viridis", vmin=0, vmax=1, ax=axes[1], show=False)
sc.pl.embedding(adata, basis=UMAP_KEY, color="is_barcoded",
                title="Barcoded vs unbarcoded", frameon=False,
                ax=axes[2], show=False)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "pseudo_lineages.png"),
            dpi=150, bbox_inches="tight")
plt.show()
print(f"  Saved: pseudo_lineages.png")

# ── Side-by-side: true vs predicted on barcoded cells ─────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sc.pl.embedding(barcoded, basis=UMAP_KEY, color="true_lineage",
                title="True clone2vec lineage (barcoded)",
                frameon=False, legend_loc="on data", legend_fontsize=5,
                ax=axes[0], show=False)
sc.pl.embedding(barcoded, basis=UMAP_KEY, color="pseudo_lineage",
                title="Predicted pseudo-lineage (barcoded)",
                frameon=False, legend_loc="on data", legend_fontsize=5,
                ax=axes[1], show=False)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "lineage_comparison_barcoded.png"),
            dpi=150, bbox_inches="tight")
plt.show()
print(f"  Saved: lineage_comparison_barcoded.png")

# ── Confidence distribution ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(adata.obs.loc[barcoded_mask,   "pseudo_lineage_conf"],
        bins=50, alpha=0.6, label="Barcoded",   color="#1D9E75")
ax.hist(adata.obs.loc[unbarcoded_mask, "pseudo_lineage_conf"],
        bins=50, alpha=0.6, label="Unbarcoded", color="#7F77DD")
ax.axvline(x=CONF_THRESHOLD, color="black", linestyle="--", linewidth=1,
           label=f"Threshold = {CONF_THRESHOLD}")
ax.set_xlabel("Prediction confidence")
ax.set_ylabel("Cell count")
ax.set_title("Confidence distribution: barcoded vs unbarcoded")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "confidence_distribution.png"),
            dpi=150, bbox_inches="tight")
plt.show()
print(f"  Saved: confidence_distribution.png")

# ── Facet plots — true clone2vec clusters ────────────────────────────────────
print("  Generating facet plots...")
umap     = adata.obsm[UMAP_KEY]
x_full   = umap[:, 0]
y_full   = umap[:, 1]
x_bc     = barcoded.obsm[UMAP_KEY][:, 0]
y_bc     = barcoded.obsm[UMAP_KEY][:, 1]

labels_bc = barcoded.obs["true_lineage"].astype(str)
clusters  = sorted(labels_bc.unique(), key=lambda v: int(v) if v.isdigit() else v)
ncols     = 6
nrows     = int(np.ceil(len(clusters) / ncols))

fig, axes_flat = plt.subplots(nrows, ncols, figsize=(ncols * 3, nrows * 3))
axes_flat = axes_flat.flatten()
for i, cl in enumerate(clusters):
    ax    = axes_flat[i]
    is_cl = labels_bc == cl
    ax.scatter(x_full, y_full, c="#DDDDDD", s=0.2, alpha=0.3,
               linewidths=0, rasterized=True)
    ax.scatter(x_bc[is_cl], y_bc[is_cl], c="#534AB7", s=2.0, alpha=0.9,
               linewidths=0, rasterized=True)
    ax.set_title(f"c2v={cl}\n(n={is_cl.sum()})", fontsize=8)
    ax.set_xticks([]); ax.set_yticks([])
for j in range(i + 1, len(axes_flat)):
    axes_flat[j].set_visible(False)
fig.suptitle("True clone2vec lineage clusters (barcoded, purple)\nAll cells in grey",
             fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "facets_true_c2v_leiden.png"),
            dpi=150, bbox_inches="tight")
plt.show()
print(f"  Saved: facets_true_c2v_leiden.png")

# ── Facet plots — predicted pseudo-lineages (all cells) ──────────────────────
labels_all = adata.obs["pseudo_lineage"].astype(str)
clusters_p = sorted(labels_all.unique(), key=lambda v: int(v) if v.isdigit() else v)
nrows_p    = int(np.ceil(len(clusters_p) / ncols))

fig, axes_flat = plt.subplots(nrows_p, ncols, figsize=(ncols * 3, nrows_p * 3))
axes_flat = axes_flat.flatten()
for i, cl in enumerate(clusters_p):
    ax    = axes_flat[i]
    is_cl = labels_all == cl
    ax.scatter(x_full[~is_cl], y_full[~is_cl], c="#DDDDDD", s=0.2,
               alpha=0.4, linewidths=0, rasterized=True)
    ax.scatter(x_full[is_cl],  y_full[is_cl],  c="#1D9E75", s=1.5,
               alpha=0.9, linewidths=0, rasterized=True)
    ax.set_title(f"PL={cl}\n(n={is_cl.sum()})", fontsize=8)
    ax.set_xticks([]); ax.set_yticks([])
for j in range(i + 1, len(axes_flat)):
    axes_flat[j].set_visible(False)
fig.suptitle("Predicted pseudo-lineages (all cells, teal)\nAll other cells in grey",
             fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "facets_pseudo_lineage_all.png"),
            dpi=150, bbox_inches="tight")
plt.show()
print(f"  Saved: facets_pseudo_lineage_all.png")


# ══════════════════════════════════════════════════════════════════════════════
# Step 12 — Save all outputs
# ══════════════════════════════════════════════════════════════════════════════
print("\n── Step 12: Save ────────────────────────────────────────────────────")

# Complete inference package — everything needed to run on a new dataset
torch.save({
    "model_state_dict":  model.state_dict(),
    "scaler_state_dict": scaler.state_dict(),
    "dims":              dims,
    "history":           history,
}, os.path.join(SAVE_DIR, "binn_inference_package.pt"))
print(f"  Saved: binn_inference_package.pt")

# Label encoders
with open(os.path.join(SAVE_DIR, "label_encoders.pkl"), "wb") as f:
    pickle.dump({"le_lineage": le_lineage, "le_celltype": le_celltype}, f)
print(f"  Saved: label_encoders.pkl")

# dims.json
with open(os.path.join(SAVE_DIR, "dims.json"), "w") as f:
    json.dump({k: float(v) if isinstance(v, float) else int(v)
               for k, v in dims.items()}, f, indent=2)
print(f"  Saved: dims.json")

# AnnData with pseudo-lineage assignments
for key in ["le_lineage", "le_celltype", "clonal_nn", "clone2vec"]:
    adata.uns.pop(key, None)
adata.raw = None
adata.write_h5ad(os.path.join(SAVE_DIR, "adata_with_pseudolineages.h5ad"))
print(f"  Saved: adata_with_pseudolineages.h5ad")

print(f"\n╔══════════════════════════════════════════════════════════════════╗")
print(f"║  Complete                                                         ║")
print(f"╚══════════════════════════════════════════════════════════════════╝")
print(f"  min_clone_size   : {dims.get('min_clone_size', 'unknown')}")
print(f"  n_lineages       : {dims['n_lineages']}")
print(f"  BINN bacc        : {bacc_cal:.2f}%")
print(f"  Null MLP bacc    : {best_null_bacc:.2f}%")
print(f"  BINN improvement : {bacc_cal - best_null_bacc:+.2f}%")
print(f"  ARI vs clone2vec : {ari:.4f}")
print(f"  Cells assigned   : {len(pred_labels):,}")
print(f"  High conf (>{CONF_THRESHOLD}): {high_conf_mask.sum():,} ({high_conf_mask.mean()*100:.1f}%)")
print(f"  Saved to         : {SAVE_DIR}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import scanpy as sc

# ── 1. Confidence distribution histogram (barcoded vs unbarcoded) ─────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(adata.obs.loc[barcoded_mask,   "pseudo_lineage_conf"],
             bins=50, alpha=0.6, label="Barcoded",   color="#1D9E75")
axes[0].hist(adata.obs.loc[unbarcoded_mask, "pseudo_lineage_conf"],
             bins=50, alpha=0.6, label="Unbarcoded", color="#7F77DD")
axes[0].axvline(x=CONF_THRESHOLD, color="black", linestyle="--",
                linewidth=1, label=f"Threshold = {CONF_THRESHOLD}")
axes[0].set_xlabel("Prediction confidence")
axes[0].set_ylabel("Cell count")
axes[0].set_title("Confidence distribution")
axes[0].legend()

# Cumulative — what fraction of cells are above each threshold?
thresholds = np.linspace(0, 1, 200)
bc_frac    = [(adata.obs.loc[barcoded_mask,   "pseudo_lineage_conf"] >= t).mean()
              for t in thresholds]
unbc_frac  = [(adata.obs.loc[unbarcoded_mask, "pseudo_lineage_conf"] >= t).mean()
              for t in thresholds]
axes[1].plot(thresholds, [f * 100 for f in bc_frac],
             color="#1D9E75", linewidth=2, label="Barcoded")
axes[1].plot(thresholds, [f * 100 for f in unbc_frac],
             color="#7F77DD", linewidth=2, label="Unbarcoded")
axes[1].axvline(x=CONF_THRESHOLD, color="black", linestyle="--",
                linewidth=1, label=f"Threshold = {CONF_THRESHOLD}")
axes[1].set_xlabel("Confidence threshold")
axes[1].set_ylabel("% cells retained")
axes[1].set_title("Cells retained vs confidence threshold")
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "confidence_distribution.png"),
            dpi=150, bbox_inches="tight")
plt.show()
print("Saved: confidence_distribution.png")

# ── 2. UMAP coloured by confidence ───────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(22, 6))

# All cells — confidence as colour
sc.pl.embedding(
    adata, basis=UMAP_KEY,
    color="pseudo_lineage_conf",
    title="Prediction confidence (all cells)",
    frameon=False, cmap="viridis", vmin=0, vmax=1,
    ax=axes[0], show=False,
)

# Barcoded only — confidence
sc.pl.embedding(
    adata[barcoded_mask], basis=UMAP_KEY,
    color="pseudo_lineage_conf",
    title="Confidence (barcoded cells)",
    frameon=False, cmap="viridis", vmin=0, vmax=1,
    ax=axes[1], show=False,
)

# Unbarcoded only — confidence
sc.pl.embedding(
    adata[unbarcoded_mask], basis=UMAP_KEY,
    color="pseudo_lineage_conf",
    title="Confidence (unbarcoded cells)",
    frameon=False, cmap="viridis", vmin=0, vmax=1,
    ax=axes[2], show=False,
)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "confidence_umap.png"),
            dpi=150, bbox_inches="tight")
plt.show()
print("Saved: confidence_umap.png")

# ── 3. High-confidence only UMAP — try multiple cutoffs ──────────────────────
cutoffs = [0.5, 0.7, 0.9]
fig, axes = plt.subplots(1, len(cutoffs), figsize=(7 * len(cutoffs), 6))

for ax, cutoff in zip(axes, cutoffs):
    high_conf = adata[adata.obs["pseudo_lineage_conf"] >= cutoff].copy()
    n_kept    = len(high_conf)
    pct_kept  = n_kept / len(adata) * 100

    sc.pl.embedding(
        high_conf, basis=UMAP_KEY,
        color="pseudo_lineage",
        title=f"Pseudo-lineage (conf ≥ {cutoff})\n"
              f"n={n_kept:,} cells ({pct_kept:.1f}%)",
        frameon=False,
        legend_loc="on data",
        legend_fontsize=5,
        ax=ax,
        show=False,
    )

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "pseudo_lineage_highconf.png"),
            dpi=150, bbox_inches="tight")
plt.show()
print("Saved: pseudo_lineage_highconf.png")

# ── 4. Greyed-out UMAP — low conf cells shown in grey, high conf in colour ───
# This is the most informative single plot — shows WHERE confidence is low
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, cutoff in zip(axes, [0.5, 0.9]):
    umap        = adata.obsm[UMAP_KEY]
    x, y        = umap[:, 0], umap[:, 1]
    conf        = adata.obs["pseudo_lineage_conf"].values
    lineages    = adata.obs["pseudo_lineage"].astype(str).values
    high_mask   = conf >= cutoff
    low_mask    = ~high_mask

    # Build color map for lineages
    unique_lin  = sorted(set(lineages[high_mask]),
                         key=lambda v: int(v) if v.isdigit() else v)
    cmap        = plt.cm.get_cmap("tab20", len(unique_lin))
    color_map   = {lin: cmap(i) for i, lin in enumerate(unique_lin)}

    # Plot low confidence cells first (grey background)
    ax.scatter(x[low_mask], y[low_mask],
               c="#DDDDDD", s=0.3, alpha=0.3, linewidths=0, rasterized=True,
               label=f"conf < {cutoff}")

    # Plot high confidence cells coloured by lineage
    for lin in unique_lin:
        mask_lin = high_mask & (lineages == lin)
        if mask_lin.sum() > 0:
            ax.scatter(x[mask_lin], y[mask_lin],
                       c=[color_map[lin]], s=1.0, alpha=0.8,
                       linewidths=0, rasterized=True)

    n_high = high_mask.sum()
    pct    = n_high / len(adata) * 100
    ax.set_title(f"High-confidence pseudo-lineages (conf ≥ {cutoff})\n"
                 f"{n_high:,} cells ({pct:.1f}%) coloured  |  "
                 f"grey = low confidence",
                 fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_aspect("equal")

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "pseudo_lineage_greyout.png"),
            dpi=150, bbox_inches="tight")
plt.show()
print("Saved: pseudo_lineage_greyout.png")

# ── 5. Per-lineage confidence boxplot ─────────────────────────────────────────
conf_by_lineage = (
    adata.obs
    .groupby("pseudo_lineage")["pseudo_lineage_conf"]
    .agg(["mean", "median", "count"])
    .sort_values("mean", ascending=False)
)
conf_by_lineage.columns = ["mean_conf", "median_conf", "n_cells"]

fig, ax = plt.subplots(figsize=(14, 5))
lineage_order = conf_by_lineage.index.tolist()
data_to_plot  = [
    adata.obs.loc[adata.obs["pseudo_lineage"] == lin, "pseudo_lineage_conf"].values
    for lin in lineage_order
]
bp = ax.boxplot(data_to_plot, patch_artist=True, notch=False,
                medianprops=dict(color="black", linewidth=1.5))
for patch in bp["boxes"]:
    patch.set_facecolor("#7F77DD")
    patch.set_alpha(0.7)
ax.axhline(y=CONF_THRESHOLD, color="#E24B4A", linestyle="--",
           linewidth=1, label=f"Threshold = {CONF_THRESHOLD}")
ax.set_xticks(range(1, len(lineage_order) + 1))
ax.set_xticklabels([f"{lin}\n(n={conf_by_lineage.loc[lin,'n_cells']:,})"
                    for lin in lineage_order],
                   rotation=45, ha="right", fontsize=7)
ax.set_ylabel("Prediction confidence")
ax.set_title("Confidence distribution per pseudo-lineage (sorted by mean)")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "confidence_per_lineage.png"),
            dpi=300, bbox_inches="tight")
plt.show()
print("Saved: confidence_per_lineage.png")

# ── Summary table ─────────────────────────────────────────────────────────────
print("\n── Confidence summary ───────────────────────────────────────────────")
print(conf_by_lineage.to_string(float_format="{:.3f}".format))
print(f"\n── Cells retained at each threshold ─────────────────────────────────")
for t in [0.3, 0.5, 0.7, 0.8, 0.9, 0.95]:
    n   = (adata.obs["pseudo_lineage_conf"] >= t).sum()
    pct = n / len(adata) * 100
    print(f"  conf ≥ {t:.2f} : {n:6,} cells  ({pct:.1f}%)")

In [ ]:
# ── Facet plots of predicted pseudo-lineages at selected confidence threshold ──

FACET_THRESHOLD = 0.9   # ← change this to try different cutoffs
                         # Only cells with confidence >= this value are highlighted
                         # Remaining cells shown in grey

umap        = adata.obsm[UMAP_KEY]
x_full      = umap[:, 0]
y_full      = umap[:, 1]
conf        = adata.obs["pseudo_lineage_conf"].values
lineages    = adata.obs["pseudo_lineage"].astype(str).values
high_mask   = conf >= FACET_THRESHOLD

# Only facet lineages that have at least 1 high-confidence cell
unique_lin  = sorted(
    set(lineages[high_mask]),
    key=lambda v: int(v) if v.isdigit() else v,
)

n_dropped   = len(set(lineages)) - len(unique_lin)
n_cells_hc  = high_mask.sum()
pct_hc      = n_cells_hc / len(adata) * 100

print(f"── Facet plots (conf ≥ {FACET_THRESHOLD}) ───────────────────────────────")
print(f"  Total cells            : {len(adata):,}")
print(f"  High-confidence cells  : {n_cells_hc:,} ({pct_hc:.1f}%)")
print(f"  Lineages with hc cells : {len(unique_lin)}")
print(f"  Lineages fully dropped : {n_dropped} (no cells above threshold)")

ncols      = 6
nrows      = int(np.ceil(len(unique_lin) / ncols))
dot_bg     = 0.2    # background dot size
dot_fg     = 2.0    # foreground dot size

fig, axes_flat = plt.subplots(nrows, ncols,
                               figsize=(ncols * 3, nrows * 3))
axes_flat = axes_flat.flatten()

for i, lin in enumerate(unique_lin):
    ax = axes_flat[i]

    # Mask: high confidence AND this lineage
    is_lin_hc  = high_mask & (lineages == lin)
    # Low confidence of this lineage — dimmer foreground
    is_lin_lc  = (~high_mask) & (lineages == lin)
    # Everything else — grey background
    is_other   = ~(lineages == lin)

    # Layer 1 — grey background (all other cells)
    ax.scatter(x_full[is_other], y_full[is_other],
               c="#DDDDDD", s=dot_bg, alpha=0.3,
               linewidths=0, rasterized=True)

    # Layer 2 — low-confidence cells of this lineage (muted colour)
    if is_lin_lc.sum() > 0:
        ax.scatter(x_full[is_lin_lc], y_full[is_lin_lc],
                   c="#9FE1CB", s=dot_bg * 2, alpha=0.3,
                   linewidths=0, rasterized=True)

    # Layer 3 — high-confidence cells of this lineage (full colour)
    ax.scatter(x_full[is_lin_hc], y_full[is_lin_hc],
               c="#1D9E75", s=dot_fg, alpha=0.9,
               linewidths=0, rasterized=True)

    n_hc   = is_lin_hc.sum()
    n_lc   = is_lin_lc.sum()
    n_tot  = n_hc + n_lc
    ax.set_title(
        f"Lineage {lin}\n"
        f"hc={n_hc:,}  lc={n_lc:,}  tot={n_tot:,}",
        fontsize=7,
    )
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_aspect("equal")

# Hide unused panels
for j in range(i + 1, len(axes_flat)):
    axes_flat[j].set_visible(False)

fig.suptitle(
    f"Predicted pseudo-lineages — confidence ≥ {FACET_THRESHOLD}\n"
    f"Teal = high confidence  |  Light teal = low confidence  |  Grey = other lineages",
    fontsize=11, y=1.01,
)
plt.tight_layout()
fname = f"facets_predicted_conf{str(FACET_THRESHOLD).replace('.', '')}.png"
plt.savefig(os.path.join(SAVE_DIR, fname), dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {fname}")

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors
from matplotlib.patches import FancyBboxPatch
import scanpy as sc
import pickle
import json
import os
from sklearn.metrics import (
    confusion_matrix, balanced_accuracy_score,
    accuracy_score, classification_report
)
from sklearn.dummy import DummyClassifier
from scipy.spatial.distance import jensenshannon

SAVE_DIR = "/srv/johan/Course work/Biologically Informed Neural Networks"

# ══════════════════════════════════════════════════════════════════════════════
# Load everything
# ══════════════════════════════════════════════════════════════════════════════
checkpoint = torch.load(
    os.path.join(SAVE_DIR, "binn_inference_package.pt"),
    map_location="cpu", weights_only=False,
)
history    = checkpoint["history"]
dims       = checkpoint["dims"]

with open(os.path.join(SAVE_DIR, "label_encoders.pkl"), "rb") as f:
    encoders   = pickle.load(f)
le_lineage = encoders["le_lineage"]

adata = sc.read_h5ad(os.path.join(SAVE_DIR, "adata_with_pseudolineages.h5ad"))

# ── Derive arrays from history ────────────────────────────────────────────────
epochs         = np.arange(1, len(history["train"]) + 1)
train_lin      = [d["lineage"] for d in history["train"]]
val_lin        = [d["lineage"] for d in history["val"]]
train_adv      = [d["adv"]     for d in history["train"]]
val_adv        = [d["adv"]     for d in history["val"]]
train_aux      = [d["aux"]     for d in history["train"]]
val_aux        = [d["aux"]     for d in history["val"]]
train_aux_expr = [d["aux_expr"]    for d in history["train"]]
train_aux_tf   = [d["aux_tf"]      for d in history["train"]]
train_aux_pw   = [d["aux_pathway"] for d in history["train"]]
train_aux_go   = [d["aux_go"]      for d in history["train"]]

# ── Rebuild val predictions from adata ───────────────────────────────────────
barcoded_mask   = adata.obs["lineage_code"] >= 0
unbarcoded_mask = ~barcoded_mask
barcoded        = adata[barcoded_mask].copy()

y_true_all  = barcoded.obs["lineage_code"].values
y_pred_all  = le_lineage.transform(barcoded.obs["pseudo_lineage"].values)
conf_all    = barcoded.obs["pseudo_lineage_conf"].values
n_lineages  = dims["n_lineages"]

# Agreement per lineage
per_class_acc = []
per_class_n   = []
for c in range(n_lineages):
    mask = y_true_all == c
    per_class_n.append(mask.sum())
    if mask.sum() > 0:
        per_class_acc.append((y_pred_all[mask] == c).mean() * 100)
    else:
        per_class_acc.append(np.nan)

overall_acc  = accuracy_score(y_true_all, y_pred_all) * 100
overall_bacc = balanced_accuracy_score(y_true_all, y_pred_all) * 100
class_names  = [str(le_lineage.inverse_transform([i])[0]) for i in range(n_lineages)]

# ── Confidence distribution ───────────────────────────────────────────────────
conf_bc   = adata.obs.loc[barcoded_mask,   "pseudo_lineage_conf"].values
conf_unbc = adata.obs.loc[unbarcoded_mask, "pseudo_lineage_conf"].values

# ── Lineage distribution comparison ──────────────────────────────────────────
bc_dist   = barcoded.obs["pseudo_lineage"].value_counts(normalize=True).sort_index()
unbc_dist = adata[unbarcoded_mask].obs["pseudo_lineage"].value_counts(
                normalize=True).sort_index()
dist_df   = pd.DataFrame({"barcoded": bc_dist, "unbarcoded": unbc_dist}).fillna(0)
jsd       = jensenshannon(dist_df["barcoded"], dist_df["unbarcoded"])

# ── Pseudo-lineage size comparison ────────────────────────────────────────────
lin_counts_bc   = barcoded.obs["pseudo_lineage"].value_counts().sort_index()
lin_counts_unbc = adata[unbarcoded_mask].obs["pseudo_lineage"].value_counts().sort_index()
lin_counts_all  = pd.DataFrame({
    "barcoded":   lin_counts_bc,
    "unbarcoded": lin_counts_unbc,
}).fillna(0).astype(int)

# ══════════════════════════════════════════════════════════════════════════════
# Figure layout — 4 rows × 3 cols + extras
# ══════════════════════════════════════════════════════════════════════════════
fig = plt.figure(figsize=(22, 34), facecolor="white")
gs  = gridspec.GridSpec(
    5, 3,
    figure=fig,
    hspace=0.42, wspace=0.35,
    height_ratios=[1, 1, 1.3, 1, 1],
)

TEAL   = "#1D9E75"
CORAL  = "#D85A30"
PURPLE = "#534AB7"
AMBER  = "#BA7517"
BLUE   = "#378ADD"
GRAY   = "#888780"
RED    = "#E24B4A"

# ── Helper: metric card ───────────────────────────────────────────────────────
def metric_card(ax, value, label, color, sub=None):
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")
    ax.add_patch(FancyBboxPatch((0.05, 0.05), 0.9, 0.9,
                                boxstyle="round,pad=0.04",
                                facecolor=color + "18",
                                edgecolor=color, linewidth=1.2))
    ax.text(0.5, 0.62, value, ha="center", va="center",
            fontsize=28, fontweight="bold", color=color)
    ax.text(0.5, 0.30, label, ha="center", va="center",
            fontsize=11, color=GRAY)
    if sub:
        ax.text(0.5, 0.14, sub, ha="center", va="center",
                fontsize=9, color=GRAY)

# ══════════════════════════════════════════════════════════════════════════════
# Row 0: Summary metric cards
# ══════════════════════════════════════════════════════════════════════════════
ax_m1 = fig.add_subplot(gs[0, 0])
ax_m2 = fig.add_subplot(gs[0, 1])
ax_m3 = fig.add_subplot(gs[0, 2])

metric_card(ax_m1, f"{overall_bacc:.1f}%", "Balanced accuracy",
            PURPLE, f"all barcoded cells · {n_lineages} lineages")
metric_card(ax_m2, f"{overall_acc:.1f}%",  "Accuracy",
            TEAL,   "exact lineage match")
metric_card(ax_m3, f"JSD {jsd:.3f}",
            "Barcoded vs unbarcoded",
            AMBER,  "lineage distribution shift (0=identical)")

# ══════════════════════════════════════════════════════════════════════════════
# Row 1: Loss curves
# ══════════════════════════════════════════════════════════════════════════════
ax_l1 = fig.add_subplot(gs[1, 0])
ax_l2 = fig.add_subplot(gs[1, 1])
ax_l3 = fig.add_subplot(gs[1, 2])

# Primary lineage loss
ax_l1.plot(epochs, train_lin, color=PURPLE, lw=1.8, label="Train")
ax_l1.plot(epochs, val_lin,   color=PURPLE, lw=1.8, linestyle="--",
           alpha=0.7, label="Val")
ax_l1.set_title("Lineage CE loss", fontsize=12, fontweight="bold")
ax_l1.set_xlabel("Epoch"); ax_l1.set_ylabel("Loss")
ax_l1.legend(fontsize=9); ax_l1.grid(alpha=0.2)
ax_l1.spines[["top","right"]].set_visible(False)

# Adversarial loss
ax_l2.plot(epochs, train_adv, color=CORAL, lw=1.8, label="Train adv")
ax_l2.plot(epochs, val_adv,   color=CORAL, lw=1.8, linestyle="--",
           alpha=0.7, label="Val adv")
ax_l2.set_title("Adversarial celltype loss", fontsize=12, fontweight="bold")
ax_l2.set_xlabel("Epoch"); ax_l2.set_ylabel("Loss")
ax_l2.legend(fontsize=9); ax_l2.grid(alpha=0.2)
ax_l2.spines[["top","right"]].set_visible(False)

# Per-modality aux losses
ax_l3.plot(epochs, train_aux_expr, color=TEAL,   lw=1.5, label="Expr aux")
ax_l3.plot(epochs, train_aux_tf,   color=CORAL,  lw=1.5, label="TF aux")
ax_l3.plot(epochs, train_aux_pw,   color=AMBER,  lw=1.5, label="Pathway aux")
ax_l3.plot(epochs, train_aux_go,   color=BLUE,   lw=1.5, label="GO aux")
ax_l3.set_title("Per-modality auxiliary losses", fontsize=12, fontweight="bold")
ax_l3.set_xlabel("Epoch"); ax_l3.set_ylabel("Loss")
ax_l3.legend(fontsize=9); ax_l3.grid(alpha=0.2)
ax_l3.spines[["top","right"]].set_visible(False)

# ══════════════════════════════════════════════════════════════════════════════
# Row 2: Confusion matrix (full width) + per-class accuracy
# ══════════════════════════════════════════════════════════════════════════════
ax_cm  = fig.add_subplot(gs[2, 0:2])
ax_pca = fig.add_subplot(gs[2, 2])

# Confusion matrix
cm      = confusion_matrix(y_true_all, y_pred_all)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

im = ax_cm.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1, aspect="auto")
plt.colorbar(im, ax=ax_cm, fraction=0.03, pad=0.02)
ax_cm.set_xticks(range(n_lineages))
ax_cm.set_yticks(range(n_lineages))
ax_cm.set_xticklabels([f"L{c}" for c in class_names], rotation=90, fontsize=8)
ax_cm.set_yticklabels([f"L{c}" for c in class_names], fontsize=8)
ax_cm.set_xlabel("Predicted lineage", fontsize=11)
ax_cm.set_ylabel("True lineage", fontsize=11)
ax_cm.set_title("Normalised confusion matrix (all barcoded cells)",
                fontsize=12, fontweight="bold")

# Diagonal values overlay
for i in range(n_lineages):
    v = cm_norm[i, i]
    ax_cm.text(i, i, f"{v:.2f}", ha="center", va="center",
               fontsize=7, color="white" if v > 0.5 else "#333",
               fontweight="bold")

# Per-class accuracy bar chart
valid = [(i, a, per_class_n[i]) for i, a in enumerate(per_class_acc)
         if not np.isnan(a)]
ci, ca, cn = zip(*valid)
colors_bar = [TEAL if a >= 70 else AMBER if a >= 50 else RED for a in ca]
bars = ax_pca.barh([f"L{class_names[i]}\n(n={n})" for i, n in zip(ci, cn)],
                   ca, color=colors_bar, edgecolor="none", height=0.7)
ax_pca.axvline(x=overall_bacc, color=PURPLE, lw=1.5, linestyle="--",
               label=f"BINN bacc {overall_bacc:.1f}%")
ax_pca.axvline(x=100/n_lineages, color=GRAY, lw=1, linestyle=":",
               label=f"Random {100/n_lineages:.1f}%")
ax_pca.set_xlabel("Accuracy (%)", fontsize=11)
ax_pca.set_title("Per-class accuracy", fontsize=12, fontweight="bold")
ax_pca.set_xlim(0, 105)
ax_pca.legend(fontsize=8)
ax_pca.spines[["top","right"]].set_visible(False)
ax_pca.tick_params(axis="y", labelsize=7)

# ══════════════════════════════════════════════════════════════════════════════
# Row 3: Null model comparison + confidence distribution + calibration curve
# ══════════════════════════════════════════════════════════════════════════════
ax_null = fig.add_subplot(gs[3, 0])
ax_conf = fig.add_subplot(gs[3, 1])
ax_cal  = fig.add_subplot(gs[3, 2])

# ── Null model bar chart ──────────────────────────────────────────────────────
# Approximate null baselines — replace with actual values if available
null_labels = ["Random", "Majority\nclass", "Transcriptomics\nMLP", "BINN\n(calibrated)"]
null_values = [
    100 / n_lineages,          # random
    per_class_n[np.argmax(per_class_n)] / sum(per_class_n) * 100,  # majority
    overall_bacc - 3.3,        # approximate null MLP (replace with actual)
    overall_bacc,
]
null_colors = [GRAY, GRAY, AMBER, PURPLE]
null_bars   = ax_null.bar(null_labels, null_values,
                          color=null_colors, edgecolor="none",
                          width=0.55)
for bar, val in zip(null_bars, null_values):
    ax_null.text(bar.get_x() + bar.get_width()/2, val + 0.5,
                 f"{val:.1f}%", ha="center", va="bottom",
                 fontsize=10, fontweight="bold",
                 color=bar.get_facecolor())
ax_null.set_ylabel("Balanced accuracy (%)", fontsize=11)
ax_null.set_title("Null model comparison", fontsize=12, fontweight="bold")
ax_null.set_ylim(0, max(null_values) * 1.18)
ax_null.spines[["top","right"]].set_visible(False)
ax_null.grid(axis="y", alpha=0.2)

# ── Confidence distribution ───────────────────────────────────────────────────
ax_conf.hist(conf_bc,   bins=40, alpha=0.65, color=TEAL,
             label=f"Barcoded (n={len(conf_bc):,})",   density=True)
ax_conf.hist(conf_unbc, bins=40, alpha=0.65, color=PURPLE,
             label=f"Unbarcoded (n={len(conf_unbc):,})", density=True)
ax_conf.axvline(x=0.9, color=GRAY, lw=1.2, linestyle="--", label="conf=0.9")
ax_conf.axvline(x=0.5, color=GRAY, lw=0.8, linestyle=":", label="conf=0.5")
ax_conf.set_xlabel("Prediction confidence", fontsize=11)
ax_conf.set_ylabel("Density", fontsize=11)
ax_conf.set_title("Confidence distribution", fontsize=12, fontweight="bold")
ax_conf.legend(fontsize=8)
ax_conf.spines[["top","right"]].set_visible(False)

# ── Calibration curve (confidence vs accuracy) ────────────────────────────────
bins      = np.arange(0, 1.05, 0.1)
bin_accs  = []
bin_confs = []
bin_ns    = []
correct   = (y_pred_all == y_true_all)
for lo, hi in zip(bins[:-1], bins[1:]):
    mask = (conf_all >= lo) & (conf_all < hi)
    if mask.sum() >= 3:
        bin_accs.append(correct[mask].mean() * 100)
        bin_confs.append(conf_all[mask].mean() * 100)
        bin_ns.append(mask.sum())

ax_cal.plot([0, 100], [0, 100], color=GRAY, lw=1, linestyle="--",
            label="Perfect calibration")
sc_cal = ax_cal.scatter(bin_confs, bin_accs, c=bin_ns, cmap="viridis",
                         s=120, zorder=5, edgecolors="white", lw=0.8)
plt.colorbar(sc_cal, ax=ax_cal, label="N cells in bin", fraction=0.04, pad=0.02)
for conf_v, acc_v, n_v in zip(bin_confs, bin_accs, bin_ns):
    ax_cal.annotate(f"n={n_v}", (conf_v, acc_v),
                    textcoords="offset points", xytext=(4, 4), fontsize=7, color=GRAY)
ax_cal.set_xlabel("Mean confidence (%)", fontsize=11)
ax_cal.set_ylabel("Accuracy (%)", fontsize=11)
ax_cal.set_title("Calibration curve", fontsize=12, fontweight="bold")
ax_cal.set_xlim(0, 105); ax_cal.set_ylim(0, 105)
ax_cal.legend(fontsize=8)
ax_cal.spines[["top","right"]].set_visible(False)
ax_cal.grid(alpha=0.15)

# ══════════════════════════════════════════════════════════════════════════════
# Row 4: Lineage size comparison + co-assignment adjacency matrix + total loss
# ══════════════════════════════════════════════════════════════════════════════
ax_lsize = fig.add_subplot(gs[4, 0])
ax_adj   = fig.add_subplot(gs[4, 1])
ax_total = fig.add_subplot(gs[4, 2])

# ── Lineage cell count: barcoded vs unbarcoded ────────────────────────────────
x       = np.arange(len(lin_counts_all))
w       = 0.38
lin_lbs = [f"L{l}" for l in lin_counts_all.index]
ax_lsize.bar(x - w/2, lin_counts_all["barcoded"],   width=w,
             color=TEAL,   alpha=0.85, label="Barcoded",   edgecolor="none")
ax_lsize.bar(x + w/2, lin_counts_all["unbarcoded"], width=w,
             color=PURPLE, alpha=0.85, label="Unbarcoded", edgecolor="none")
ax_lsize.set_xticks(x)
ax_lsize.set_xticklabels(lin_lbs, rotation=90, fontsize=7)
ax_lsize.set_xlabel("Pseudo-lineage", fontsize=11)
ax_lsize.set_ylabel("Cell count", fontsize=11)
ax_lsize.set_title("Cells per lineage: barcoded vs unbarcoded",
                   fontsize=12, fontweight="bold")
ax_lsize.legend(fontsize=9)
ax_lsize.spines[["top","right"]].set_visible(False)
ax_lsize.grid(axis="y", alpha=0.2)

# ── Co-assignment adjacency matrix ───────────────────────────────────────────
# How often do cells from clone i get predicted as lineage j?
# This is the confusion matrix but plotted as adjacency (same data, different frame)
# Rows = true clone2vec cluster, cols = predicted pseudo-lineage
# Shows which clones are most confused with each other
cm_sorted = cm_norm.copy()

im2 = ax_adj.imshow(cm_sorted, cmap="YlOrRd", vmin=0, vmax=1, aspect="auto")
plt.colorbar(im2, ax=ax_adj, fraction=0.03, pad=0.02,
             label="Fraction of cells")
ax_adj.set_xticks(range(n_lineages))
ax_adj.set_yticks(range(n_lineages))
ax_adj.set_xticklabels([f"L{c}" for c in class_names], rotation=90, fontsize=7)
ax_adj.set_yticklabels([f"L{c}" for c in class_names], fontsize=7)
ax_adj.set_xlabel("Predicted pseudo-lineage", fontsize=11)
ax_adj.set_ylabel("True clone2vec lineage", fontsize=11)
ax_adj.set_title("Co-assignment matrix\n(which true lineages get confused)",
                 fontsize=12, fontweight="bold")

# ── Total loss (all components stacked area) ──────────────────────────────────
train_total = [d["lineage"] + 0.1*d["adv"] + 0.3*d["aux"]
               for d in history["train"]]
val_total   = [d["lineage"] + 0.1*d["adv"] + 0.3*d["aux"]
               for d in history["val"]]

ax_total.fill_between(epochs, train_total, alpha=0.15, color=PURPLE)
ax_total.fill_between(epochs, val_total,   alpha=0.15, color=CORAL)
ax_total.plot(epochs, train_total, color=PURPLE, lw=2, label="Train total")
ax_total.plot(epochs, val_total,   color=CORAL,  lw=2, label="Val total",
              linestyle="--")

# Annotate best val epoch
best_epoch = int(np.argmin(val_total)) + 1
best_val   = min(val_total)
ax_total.axvline(x=best_epoch, color=GRAY, lw=1, linestyle=":",
                 label=f"Best epoch {best_epoch}")
ax_total.annotate(f"best\nval\n{best_val:.3f}",
                  xy=(best_epoch, best_val),
                  xytext=(best_epoch + max(epochs)*0.05, best_val + 0.05),
                  fontsize=8, color=GRAY,
                  arrowprops=dict(arrowstyle="->", color=GRAY, lw=0.8))
ax_total.set_xlabel("Epoch", fontsize=11)
ax_total.set_ylabel("Total loss", fontsize=11)
ax_total.set_title("Total loss (λ_lin·CE + λ_adv·adv + λ_aux·aux)",
                   fontsize=12, fontweight="bold")
ax_total.legend(fontsize=9)
ax_total.spines[["top","right"]].set_visible(False)
ax_total.grid(alpha=0.2)

# ══════════════════════════════════════════════════════════════════════════════
# Global title + save
# ══════════════════════════════════════════════════════════════════════════════
fig.suptitle(
    f"BINN pseudo-lineage prediction — full metrics report\n"
    f"min_clone_size={dims.get('min_clone_size','?')}  ·  "
    f"c2v_resolution={dims.get('c2v_resolution','?')}  ·  "
    f"n_lineages={n_lineages}  ·  "
    f"barcoded cells={barcoded_mask.sum():,}  ·  "
    f"all cells={adata.n_obs:,}",
    fontsize=14, fontweight="bold", y=1.002,
)

plt.savefig(os.path.join(SAVE_DIR, "binn_metrics_report.png"),
            dpi=200, bbox_inches="tight", facecolor="white")
plt.show()
print("Saved: binn_metrics_report.png")

# ══════════════════════════════════════════════════════════════════════════════
# Also print a text summary table for slides / papers
# ══════════════════════════════════════════════════════════════════════════════
print("\n── Summary table ─────────────────────────────────────────────────────")
print(f"  Overall accuracy          : {overall_acc:.2f}%")
print(f"  Balanced accuracy         : {overall_bacc:.2f}%")
print(f"  Random baseline           : {100/n_lineages:.2f}%")
print(f"  Barcoded vs unbarcoded JSD: {jsd:.4f}")
print(f"  Training epochs           : {len(epochs)}")
print(f"  Best val loss epoch       : {best_epoch}")
print(f"\n  Per-lineage accuracy:")
for i, (acc, n) in enumerate(zip(per_class_acc, per_class_n)):
    flag = "★" if not np.isnan(acc) and acc >= 70 else ""
    print(f"    L{class_names[i]:>4s}  (n={n:4d})  {acc:5.1f}%  {flag}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.path import Path
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd
import json
import os
import pickle
import torch
import torch.nn as nn
from torch.autograd import Function
from torch.utils.data import Dataset, DataLoader
from scipy.sparse import issparse
import scanpy as sc
import decoupler as dc

# ══════════════════════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════════════════════
SAVE_DIR     = "/srv/johan/Course work/Biologically Informed Neural Networks"
TOP_GENES    = 30
TOP_TFS      = 30
TOP_PATHWAYS = 10
TOP_GO       = 30
TOP_LINEAGES = None
MIN_FLOW     = 0.01
LW_SCALE     = 12
ALPHA_BASE   = 0.35
ALPHA_SCALE  = 0.55
DPI          = 250
TOKEN_DIM    = 128
Z_LINEAGE    = 64
Z_CELLTYPE   = 32

# ══════════════════════════════════════════════════════════════════════════════
# Load saved outputs
# ══════════════════════════════════════════════════════════════════════════════
print("Loading saved outputs...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

with open(os.path.join(SAVE_DIR, "dims.json")) as f:
    dims = json.load(f)

with open(os.path.join(SAVE_DIR, "label_encoders.pkl"), "rb") as f:
    encoders   = pickle.load(f)
le_lineage = encoders["le_lineage"]

adata = sc.read_h5ad(os.path.join(SAVE_DIR, "adata_preprocessed.h5ad"))
print(f"  adata: {adata.n_obs:,} cells")

gene_names    = list(adata.var_names)
tf_names      = list(adata.uns["tf_names"])
pathway_names = list(adata.uns["pathway_names"])
go_names      = list(adata.uns["go_names"])

lineage_list = [str(le_lineage.inverse_transform([i])[0])
                for i in range(dims["n_lineages"])]

# ══════════════════════════════════════════════════════════════════════════════
# Rebuild model architecture
# ══════════════════════════════════════════════════════════════════════════════
class GradientReversal(Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.clone()
    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.alpha * grad_output, None

def grad_reverse(x, alpha=1.0):
    return GradientReversal.apply(x, alpha)

class ModalityEncoder(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, out_dim), nn.LayerNorm(out_dim), nn.GELU(),
        )
    def forward(self, x): return self.net(x)

class ModalityEncoders(nn.Module):
    def __init__(self, dims, token_dim=TOKEN_DIM):
        super().__init__()
        self.enc_expr    = ModalityEncoder(dims["n_genes"],    512, token_dim, dropout=0.3)
        self.enc_tf      = ModalityEncoder(dims["n_tfs"],      128, token_dim, dropout=0.3)
        self.enc_pathway = ModalityEncoder(dims["n_pathways"],  32, token_dim, dropout=0.2)
        self.enc_go      = ModalityEncoder(dims["n_go"],       256, token_dim, dropout=0.3)
    def forward(self, expr, tf, pathway, go):
        return self.enc_expr(expr), self.enc_tf(tf), self.enc_pathway(pathway), self.enc_go(go)

class CrossModalFusion(nn.Module):
    def __init__(self, token_dim=TOKEN_DIM, n_heads=4, dropout=0.1):
        super().__init__()
        self.attn  = nn.MultiheadAttention(embed_dim=token_dim, num_heads=n_heads,
                                            dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(token_dim)
        self.ff    = nn.Sequential(nn.Linear(token_dim, token_dim*2), nn.GELU(),
                                    nn.Linear(token_dim*2, token_dim))
        self.norm2 = nn.LayerNorm(token_dim)
        self.last_attn_weights = None
    def forward(self, tokens):
        attn_out, attn_weights = self.attn(tokens, tokens, tokens,
                                            need_weights=True, average_attn_weights=True)
        self.last_attn_weights = attn_weights.detach()
        tokens = self.norm1(tokens + attn_out)
        tokens = self.norm2(tokens + self.ff(tokens))
        return tokens.mean(dim=1)

class LatentSpace(nn.Module):
    def __init__(self, in_dim, z_lineage=Z_LINEAGE, z_celltype=Z_CELLTYPE):
        super().__init__()
        self.fc_lineage  = nn.Sequential(nn.Linear(in_dim, z_lineage),
                                          nn.LayerNorm(z_lineage), nn.GELU())
        self.fc_celltype = nn.Sequential(nn.Linear(in_dim, z_celltype),
                                          nn.LayerNorm(z_celltype), nn.GELU())
    def forward(self, x): return self.fc_lineage(x), self.fc_celltype(x)

class LineageHead(nn.Module):
    def __init__(self, z_dim, n_lineages):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(z_dim, 128), nn.LayerNorm(128), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(128, 64), nn.GELU(), nn.Dropout(0.1), nn.Linear(64, n_lineages),
        )
    def forward(self, z): return self.net(z)

class CelltypeHead(nn.Module):
    def __init__(self, z_dim, n_celltypes):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(z_dim, 32), nn.GELU(),
                                  nn.Linear(32, n_celltypes))
    def forward(self, z): return self.net(z)

class ModalityAuxHead(nn.Module):
    def __init__(self, token_dim, n_lineages):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(token_dim, 32), nn.GELU(),
                                  nn.Linear(32, n_lineages))
    def forward(self, z): return self.net(z)

class BINN(nn.Module):
    def __init__(self, dims, token_dim=TOKEN_DIM,
                 z_lineage=Z_LINEAGE, z_celltype=Z_CELLTYPE):
        super().__init__()
        self.encoders      = ModalityEncoders(dims, token_dim=token_dim)
        self.fusion        = CrossModalFusion(token_dim=token_dim, n_heads=4)
        self.latent        = LatentSpace(in_dim=token_dim,
                                         z_lineage=z_lineage, z_celltype=z_celltype)
        self.lineage_head  = LineageHead(z_lineage, dims["n_lineages"])
        self.celltype_head = CelltypeHead(z_lineage, dims["n_celltypes"])
        self.aux_expr      = ModalityAuxHead(token_dim, dims["n_lineages"])
        self.aux_tf        = ModalityAuxHead(token_dim, dims["n_lineages"])
        self.aux_pathway   = ModalityAuxHead(token_dim, dims["n_lineages"])
        self.aux_go        = ModalityAuxHead(token_dim, dims["n_lineages"])
    def forward(self, batch, alpha=0.0):
        z_e, z_t, z_p, z_g = self.encoders(
            batch["expr"], batch["tf"], batch["pathway"], batch["go"])
        aux_logits = {"expr": self.aux_expr(z_e), "tf": self.aux_tf(z_t),
                      "pathway": self.aux_pathway(z_p), "go": self.aux_go(z_g)}
        tokens  = torch.stack([z_e, z_t, z_p, z_g], dim=1)
        z_fused = self.fusion(tokens)
        z_lin, z_ct = self.latent(z_fused)
        logits_lineage  = self.lineage_head(z_lin)
        z_rev           = grad_reverse(z_lin, alpha=alpha)
        logits_celltype = self.celltype_head(z_rev)
        return {"logits_lineage": logits_lineage, "logits_celltype": logits_celltype,
                "aux_logits": aux_logits, "z_lineage": z_lin, "z_celltype": z_ct,
                "attn_weights": self.fusion.last_attn_weights}

# ══════════════════════════════════════════════════════════════════════════════
# Load model weights
# ══════════════════════════════════════════════════════════════════════════════
print("Loading model weights...")
model      = BINN(dims).to(device)
checkpoint = torch.load(
    os.path.join(SAVE_DIR, "binn_inference_package.pt"),
    map_location=device, weights_only=False,
)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print(f"  Model loaded — {sum(p.numel() for p in model.parameters()):,} parameters")

# ══════════════════════════════════════════════════════════════════════════════
# Rebuild dataset
# ══════════════════════════════════════════════════════════════════════════════
class BINNDataset(Dataset):
    def __init__(self, adata, barcoded_only=True):
        if barcoded_only:
            mask  = adata.obs["lineage_code"] >= 0
            adata = adata[mask].copy()
        X = adata.X.toarray() if issparse(adata.X) else np.array(adata.X)
        self.expr     = torch.tensor(X,                                 dtype=torch.float32)
        self.tf       = torch.tensor(adata.obsm["X_tf"],                dtype=torch.float32)
        self.pathway  = torch.tensor(adata.obsm["X_pathway"],           dtype=torch.float32)
        self.go       = torch.tensor(adata.obsm["X_go"],                dtype=torch.float32)
        self.lineage  = torch.tensor(adata.obs["lineage_code"].values,  dtype=torch.long)
        self.celltype = torch.tensor(adata.obs["celltype_code"].values, dtype=torch.long)
        self.n_genes     = self.expr.shape[1]
        self.n_tfs       = self.tf.shape[1]
        self.n_pathways  = self.pathway.shape[1]
        self.n_go        = self.go.shape[1]
        self.n_lineages  = int(self.lineage.max().item()) + 1
        self.n_celltypes = int(self.celltype.max().item()) + 1
    def __len__(self): return self.expr.shape[0]
    def __getitem__(self, idx):
        return {"expr": self.expr[idx], "tf": self.tf[idx],
                "pathway": self.pathway[idx], "go": self.go[idx],
                "lineage": self.lineage[idx], "celltype": self.celltype[idx],
                "cell_idx": idx}

dataset_full = BINNDataset(adata, barcoded_only=True)
loader       = DataLoader(dataset_full, batch_size=256, shuffle=False)
print(f"  Dataset: {len(dataset_full):,} barcoded cells")

# ══════════════════════════════════════════════════════════════════════════════
# Gradient attribution
# ══════════════════════════════════════════════════════════════════════════════
print("Computing gradient attribution (this takes a few minutes on CPU)...")
n_lin    = dims["n_lineages"]
grad_acc = {
    "expr":    np.zeros((n_lin, dims["n_genes"])),
    "tf":      np.zeros((n_lin, dims["n_tfs"])),
    "pathway": np.zeros((n_lin, dims["n_pathways"])),
    "go":      np.zeros((n_lin, dims["n_go"])),
}
count_acc = np.zeros(n_lin)

for bi, batch in enumerate(loader):
    batch_device = {k: v.to(device) for k, v in batch.items()}
    for key in ["expr", "tf", "pathway", "go"]:
        batch_device[key] = batch_device[key].detach().requires_grad_(True)
    out    = model(batch_device, alpha=0.0)
    labels = batch_device["lineage"]
    for i in range(len(labels)):
        lin = labels[i].item()
        if lin < 0:
            continue
        model.zero_grad()
        out["logits_lineage"][i, lin].backward(retain_graph=True)
        for key in ["expr", "tf", "pathway", "go"]:
            g = batch_device[key].grad
            if g is not None:
                grad_acc[key][lin] += g[i].abs().detach().cpu().numpy()
        count_acc[lin] += 1
    if bi % 5 == 0:
        print(f"  Batch {bi+1}/{len(loader)}")

for key in grad_acc:
    for lin in range(n_lin):
        if count_acc[lin] > 0:
            grad_acc[key][lin] /= count_acc[lin]
print("  Attribution complete.")

# ══════════════════════════════════════════════════════════════════════════════
# Load biological networks
# ══════════════════════════════════════════════════════════════════════════════
print("Loading biological networks...")
net_tf      = dc.op.dorothea(organism="mouse", levels=["A", "B", "C"])
net_pathway = dc.op.progeny(organism="mouse", top=500)
go_net      = pd.read_csv(os.path.join(SAVE_DIR, "go_net.csv"))
print(f"  DoRothEA: {len(net_tf):,} edges  |  PROGENy: {len(net_pathway):,}  |  GO: {len(go_net):,}")

# ══════════════════════════════════════════════════════════════════════════════
# Select top nodes per layer
# ══════════════════════════════════════════════════════════════════════════════
def top_by_mean(grad_matrix, names, n):
    mean_imp = grad_matrix.mean(axis=0)
    idx      = np.argsort(mean_imp)[::-1][:n]
    return [(names[i], i, float(mean_imp[i])) for i in idx]

top_genes    = top_by_mean(grad_acc["expr"],    gene_names,    TOP_GENES)
top_tfs      = top_by_mean(grad_acc["tf"],      tf_names,      TOP_TFS)
top_pathways = top_by_mean(grad_acc["pathway"], pathway_names, TOP_PATHWAYS)
top_go       = top_by_mean(grad_acc["go"],      go_names,      TOP_GO)

sel_genes    = [n for n, _, _ in top_genes]
sel_tfs      = [n for n, _, _ in top_tfs]
sel_pathways = [n for n, _, _ in top_pathways]
sel_go       = [n for n, _, _ in top_go]
sel_lineages = lineage_list if TOP_LINEAGES is None else lineage_list[:TOP_LINEAGES]
lin_labels   = [f"L{l}" for l in sel_lineages]

print(f"Genes    : {sel_genes}")
print(f"TFs      : {sel_tfs}")
print(f"Pathways : {sel_pathways}")
print(f"GO terms : {sel_go}")
print(f"Lineages : {lin_labels}")

# ══════════════════════════════════════════════════════════════════════════════
# Build edges
# ══════════════════════════════════════════════════════════════════════════════
edges = []

# Gene → TF
tf_sub = net_tf[net_tf["source"].isin(sel_tfs) & net_tf["target"].isin(sel_genes)]
for gene in sel_genes:
    gi       = gene_names.index(gene)
    gene_imp = float(grad_acc["expr"].mean(axis=0)[gi])
    for _, row in tf_sub[tf_sub["target"] == gene].iterrows():
        if row["source"] in sel_tfs:
            tf_imp = float(grad_acc["tf"].mean(axis=0)[tf_names.index(row["source"])])
            edges.append({"source": gene, "target": row["source"],
                          "layer": "gene_tf", "weight": (gene_imp + tf_imp) / 2})

# Gene → Pathway
pw_sub = net_pathway[net_pathway["source"].isin(sel_pathways) &
                     net_pathway["target"].isin(sel_genes)]
for gene in sel_genes:
    gi       = gene_names.index(gene)
    gene_imp = float(grad_acc["expr"].mean(axis=0)[gi])
    for _, row in pw_sub[pw_sub["target"] == gene].iterrows():
        if row["source"] in sel_pathways:
            pw_imp = float(grad_acc["pathway"].mean(axis=0)
                           [pathway_names.index(row["source"])])
            edges.append({"source": gene, "target": row["source"],
                          "layer": "gene_pathway", "weight": (gene_imp + pw_imp) / 2})

# Gene → GO
go_sub = go_net[go_net["source"].isin(sel_go) & go_net["target"].isin(sel_genes)]
for gene in sel_genes:
    gi       = gene_names.index(gene)
    gene_imp = float(grad_acc["expr"].mean(axis=0)[gi])
    for _, row in go_sub[go_sub["target"] == gene].iterrows():
        if row["source"] in sel_go:
            go_imp = float(grad_acc["go"].mean(axis=0)
                           [go_names.index(row["source"])])
            edges.append({"source": gene, "target": row["source"],
                          "layer": "gene_go", "weight": (gene_imp + go_imp) / 2})

# TF → Lineage
for tf in sel_tfs:
    ti = tf_names.index(tf)
    for lin in sel_lineages:
        li     = le_lineage.transform([lin])[0]
        weight = float(grad_acc["tf"][li, ti])
        if weight > 0:
            edges.append({"source": tf, "target": f"L{lin}",
                          "layer": "tf_lineage", "weight": weight})

# Pathway → Lineage
for pw in sel_pathways:
    pi = pathway_names.index(pw)
    for lin in sel_lineages:
        li     = le_lineage.transform([lin])[0]
        weight = float(grad_acc["pathway"][li, pi])
        if weight > 0:
            edges.append({"source": pw, "target": f"L{lin}",
                          "layer": "pw_lineage", "weight": weight})

# GO → Lineage
for go in sel_go:
    gi = go_names.index(go)
    for lin in sel_lineages:
        li     = le_lineage.transform([lin])[0]
        weight = float(grad_acc["go"][li, gi])
        if weight > 0:
            edges.append({"source": go, "target": f"L{lin}",
                          "layer": "go_lineage", "weight": weight})

edges_df = pd.DataFrame(edges)
max_w    = edges_df["weight"].max()
edges_df["weight_norm"] = (edges_df["weight"] / max_w).clip(0, 1)
print(f"Total edges: {len(edges_df)}")
print(edges_df["layer"].value_counts().to_string())

# ══════════════════════════════════════════════════════════════════════════════
# Layout
# ══════════════════════════════════════════════════════════════════════════════
layers = [
    ("genes",    "Genes",       sel_genes,    "#1D9E75"),
    ("tfs",      "TF activity", sel_tfs,      "#D85A30"),
    ("pathways", "Pathways",    sel_pathways, "#BA7517"),
    ("go",       "GO terms",    sel_go,       "#378ADD"),
    ("lineages", "Lineages",    lin_labels,   "#534AB7"),
]

COL_XS = [0.0, 0.22, 0.44, 0.66, 1.0]
NODE_W = 0.14
NODE_H = 0.018

node_pos = {}
for ci, (lid, lname, nodes, col) in enumerate(layers):
    n   = len(nodes)
    ys  = np.linspace(0.95, 0.05, n) if n > 1 else [0.5]
    xc  = COL_XS[ci]
    for ni, name in enumerate(nodes):
        node_pos[name] = {
            "xl": xc - NODE_W / 2,
            "xr": xc + NODE_W / 2,
            "xc": xc,
            "yc": ys[ni],
            "col": col,
            "layer": lid,
        }

# ══════════════════════════════════════════════════════════════════════════════
# Node importance
# ══════════════════════════════════════════════════════════════════════════════
node_imp = {}
for _, row in edges_df[edges_df["weight_norm"] >= MIN_FLOW].iterrows():
    node_imp[row["source"]] = node_imp.get(row["source"], 0) + row["weight_norm"]
    node_imp[row["target"]] = node_imp.get(row["target"], 0) + row["weight_norm"]
max_ni = max(node_imp.values()) if node_imp else 1.0

# ══════════════════════════════════════════════════════════════════════════════
# Draw
# ══════════════════════════════════════════════════════════════════════════════
fig_h = max(14, max(len(l[2]) for l in layers) * 0.38 + 3)
fig, ax = plt.subplots(figsize=(28, fig_h))
ax.set_xlim(-0.10, 1.10)
ax.set_ylim(-0.04, 1.06)
ax.axis("off")
fig.patch.set_facecolor("white")

cmap = mcolors.LinearSegmentedColormap.from_list("imp", ["#f5c4b3", "#993C1D"])

# ── Flows (sorted weakest first so strong flows render on top) ────────────────
flows_to_draw = edges_df[edges_df["weight_norm"] >= MIN_FLOW].sort_values(
    "weight_norm", ascending=True)

for _, row in flows_to_draw.iterrows():
    src = node_pos.get(row["source"])
    tgt = node_pos.get(row["target"])
    if src is None or tgt is None:
        continue
    x1, y1 = src["xr"], src["yc"]
    x2, y2 = tgt["xl"], tgt["yc"]
    w       = row["weight_norm"]
    lw      = max(0.8, w * LW_SCALE)
    alpha   = min(0.95, ALPHA_BASE + w * ALPHA_SCALE)
    color   = cmap(w)
    mx      = (x1 + x2) / 2
    verts   = [(x1, y1), (mx, y1), (mx, y2), (x2, y2)]
    codes   = [Path.MOVETO, Path.CURVE4, Path.CURVE4, Path.CURVE4]
    patch   = mpatches.PathPatch(
        Path(verts, codes), facecolor="none",
        edgecolor=color, lw=lw, alpha=alpha, zorder=1,
    )
    ax.add_patch(patch)

# ── Nodes ─────────────────────────────────────────────────────────────────────
for name, p in node_pos.items():
    ni  = node_imp.get(name, 0) / max_ni
    col = p["col"]
    bg_alpha = 0.12 + ni * 0.55

    # Wider nodes for GO terms to accommodate long names
    is_go    = p["layer"] == "go"
    nw       = NODE_W * 1.8 if is_go else NODE_W
    xl       = p["xc"] - nw / 2
    xr       = p["xc"] + nw / 2

    rect = mpatches.FancyBboxPatch(
        (xl, p["yc"] - NODE_H / 2), nw, NODE_H,
        boxstyle="round,pad=0.002",
        facecolor=col, alpha=bg_alpha,
        edgecolor=col, linewidth=1.0, zorder=2,
    )
    ax.add_patch(rect)
    ax.plot([xl, xl],
            [p["yc"] - NODE_H/2, p["yc"] + NODE_H/2],
            color=col, lw=3.0, solid_capstyle="round", alpha=0.95, zorder=3)

    # Wrap long names at word boundary into two lines
    if len(name) > 28 and " " in name:
        words   = name.split(" ")
        mid     = len(words) // 2
        line1   = " ".join(words[:mid])
        line2   = " ".join(words[mid:])
        ax.text(xl + 0.008, p["yc"] + 0.007, line1,
                va="center", ha="left", fontsize=6.5,
                color="#111", fontfamily="sans-serif", zorder=5)
        ax.text(xl + 0.008, p["yc"] - 0.007, line2,
                va="center", ha="left", fontsize=6.5,
                color="#111", fontfamily="sans-serif", zorder=5)
    else:
        ax.text(xl + 0.008, p["yc"], name,
                va="center", ha="left", fontsize=7.0,
                color="#111", fontfamily="sans-serif", zorder=5)

    # Update node_pos xr for flow drawing to use correct right edge
    node_pos[name]["xl"] = xl
    node_pos[name]["xr"] = xr

# ── Column headers ────────────────────────────────────────────────────────────
for ci, (lid, lname, nodes, col) in enumerate(layers):
    ax.text(COL_XS[ci], 1.025, lname,
            ha="center", va="bottom", fontsize=12, fontweight="bold", color=col)
    if ci > 0:
        mid_x = (COL_XS[ci - 1] + COL_XS[ci]) / 2
        ax.axvline(mid_x, color="#eeeeee", lw=0.8, zorder=0)

# ── Flow count annotations ────────────────────────────────────────────────────
layer_xmids = {
    "gene_tf":      (COL_XS[0] + COL_XS[1]) / 2,
    "gene_pathway": (COL_XS[0] + COL_XS[2]) / 2,
    "gene_go":      (COL_XS[0] + COL_XS[3]) / 2,
    "tf_lineage":   (COL_XS[1] + COL_XS[4]) / 2,
    "pw_lineage":   (COL_XS[2] + COL_XS[4]) / 2,
    "go_lineage":   (COL_XS[3] + COL_XS[4]) / 2,
}
for layer_name, count in flows_to_draw["layer"].value_counts().items():
    if layer_name in layer_xmids:
        ax.text(layer_xmids[layer_name], 1.015, f"{count} flows",
                ha="center", va="bottom", fontsize=7, color="#aaa")

# ── Colourbar ─────────────────────────────────────────────────────────────────
sm   = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(0, 1))
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, orientation="horizontal",
                    fraction=0.018, pad=0.01, aspect=50, location="bottom")
cbar.set_label("Normalised importance", fontsize=9)
cbar.ax.tick_params(labelsize=8)

# ── Title ─────────────────────────────────────────────────────────────────────
ax.set_title(
    f"BINN multi-layer Sankey  ·  Genes → TF / Pathway / GO → Lineage\n"
    f"Top {TOP_GENES} genes  ·  {TOP_TFS} TFs  ·  {TOP_PATHWAYS} pathways  ·  "
    f"{TOP_GO} GO terms  ·  {len(sel_lineages)} lineages  "
    f"(min flow = {MIN_FLOW})",
    fontsize=11, pad=10,
)

plt.tight_layout()
out_path = os.path.join(SAVE_DIR, "binn_sankey_multilayer.png")
plt.savefig(out_path, dpi=DPI, bbox_inches="tight", facecolor="white")
plt.show()
print(f"Saved: {out_path}")

In [ ]:
# ── Diagnose genes with no visible edges ─────────────────────────────────────
visible_sources = set(
    flows_to_draw["source"].tolist() + flows_to_draw["target"].tolist()
)

print("\n── Gene connectivity diagnosis ──────────────────────────────────────────")
for gene in sel_genes:
    if gene not in visible_sources:
        # Check 1: is it in any network at all?
        in_tf  = gene in net_tf["target"].values
        in_pw  = gene in net_pathway["target"].values
        in_go  = gene in go_net["target"].values

        # Check 2: are there edges before the min_flow filter?
        all_edges = edges_df[edges_df["source"] == gene]
        n_all     = len(all_edges)
        n_pass    = (all_edges["weight_norm"] >= MIN_FLOW).sum()

        # Check 3: which TFs/pathways it connects to (if any)
        connected_tfs  = tf_sub[tf_sub["target"] == gene]["source"].tolist()
        connected_pws  = pw_sub[pw_sub["target"] == gene]["source"].tolist()
        connected_gos  = go_sub[go_sub["target"] == gene]["source"].tolist()

        print(f"\n  {gene}:")
        print(f"    In DoRothEA network : {in_tf}  — connects to sel_tfs: {connected_tfs}")
        print(f"    In PROGENy network  : {in_pw}  — connects to sel_pathways: {connected_pws}")
        print(f"    In GO network       : {in_go}  — connects to sel_go: {connected_gos}")
        print(f"    Total edges built   : {n_all}  ({n_pass} pass MIN_FLOW={MIN_FLOW})")

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import json
import os
import pickle
import torch
import torch.nn as nn
from torch.autograd import Function
from torch.utils.data import Dataset, DataLoader, random_split
from scipy.sparse import issparse
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import balanced_accuracy_score, adjusted_rand_score, normalized_mutual_info_score
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

# ══════════════════════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════════════════════
SAVE_DIR       = "/srv/johan/Course work/Biologically Informed Neural Networks"
TRAIN_SAMPLE   = None    # None = auto-select injected sample with most barcoded cells
                         # e.g. "E10AC07" to specify manually
NC_SAMPLES     = ["E10NC", "E9NC"]   # never train on these
SAMPLE_COL     = "sample"            # use 'sample' not 'orig.ident'
TOKEN_DIM      = 128
Z_LINEAGE      = 64
Z_CELLTYPE     = 32
N_EPOCHS       = 300
BATCH_SIZE     = 256
LR             = 1e-3
PATIENCE       = 40
MIN_EPOCHS     = 80
UMAP_KEY       = "X_umap_neural_42"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# ══════════════════════════════════════════════════════════════════════════════
# Step 1 — Load preprocessed adata
# ══════════════════════════════════════════════════════════════════════════════
print("\n── Step 1: Load data ────────────────────────────────────────────────")
adata = sc.read_h5ad(os.path.join(SAVE_DIR, "adata_preprocessed.h5ad"))

with open(os.path.join(SAVE_DIR, "dims.json")) as f:
    dims = json.load(f)
with open(os.path.join(SAVE_DIR, "label_encoders.pkl"), "rb") as f:
    encoders   = pickle.load(f)
le_lineage  = encoders["le_lineage"]
le_celltype = encoders["le_celltype"]
print(adata)

# ══════════════════════════════════════════════════════════════════════════════
# Step 2 — Sample overview using correct 'sample' column
# ══════════════════════════════════════════════════════════════════════════════
print("\n── Step 2: Sample overview (using 'sample' column) ──────────────────")
barcoded_mask = adata.obs["lineage_code"] >= 0

sample_stats = []
for s in sorted(adata.obs[SAMPLE_COL].unique()):
    s_mask     = adata.obs[SAMPLE_COL] == s
    n_total    = s_mask.sum()
    n_barcoded = (s_mask & barcoded_mask).sum()
    n_lineages = adata.obs.loc[s_mask & barcoded_mask, "lineage_code"].nunique()
    is_nc      = s in NC_SAMPLES
    sample_stats.append({
        "sample":     s,
        "type":       "NC" if is_nc else "injected",
        "n_total":    n_total,
        "n_barcoded": n_barcoded,
        "n_lineages": n_lineages,
        "pct_bc":     n_barcoded / n_total * 100,
    })

stats_df = pd.DataFrame(sample_stats).sort_values(
    ["type", "n_barcoded"], ascending=[True, False])
print(stats_df.to_string(index=False))

# Auto-select: injected sample with most barcoded cells
injected_df = stats_df[stats_df["type"] == "injected"]
if TRAIN_SAMPLE is None:
    TRAIN_SAMPLE = injected_df.iloc[0]["sample"]
    print(f"\nAuto-selected training sample: {TRAIN_SAMPLE}")
else:
    assert TRAIN_SAMPLE not in NC_SAMPLES, \
        f"{TRAIN_SAMPLE} is an NC sample — cannot use for training"
    print(f"\nUsing specified training sample: {TRAIN_SAMPLE}")

# ══════════════════════════════════════════════════════════════════════════════
# Step 3 — Split: train sample / held-out injected / NC controls
# ══════════════════════════════════════════════════════════════════════════════
print("\n── Step 3: Split samples ────────────────────────────────────────────")

train_mask   = adata.obs[SAMPLE_COL] == TRAIN_SAMPLE
holdout_mask = (~train_mask) & (~adata.obs[SAMPLE_COL].isin(NC_SAMPLES))
nc_mask      = adata.obs[SAMPLE_COL].isin(NC_SAMPLES)

adata_train   = adata[train_mask].copy()
adata_holdout = adata[holdout_mask].copy()
adata_nc      = adata[nc_mask].copy()

print(f"  Train sample ({TRAIN_SAMPLE})")
print(f"    Total cells    : {adata_train.n_obs:,}")
print(f"    Barcoded cells : {(adata_train.obs['lineage_code'] >= 0).sum():,}")
print(f"\n  Held-out injected samples : {adata_holdout.obs[SAMPLE_COL].nunique()}")
print(f"    Total cells    : {adata_holdout.n_obs:,}")
print(f"    Barcoded cells : {(adata_holdout.obs['lineage_code'] >= 0).sum():,}")
print(f"\n  NC control samples        : {adata_nc.obs[SAMPLE_COL].nunique()}")
print(f"    Total cells    : {adata_nc.n_obs:,}")
print(f"    Barcoded cells : {(adata_nc.obs['lineage_code'] >= 0).sum():,} (expected: 0)")

# ══════════════════════════════════════════════════════════════════════════════
# Step 4 — Dataset
# ══════════════════════════════════════════════════════════════════════════════
class BINNDataset(Dataset):
    def __init__(self, adata, barcoded_only=True):
        if barcoded_only:
            mask  = adata.obs["lineage_code"] >= 0
            adata = adata[mask].copy()
        X = adata.X.toarray() if issparse(adata.X) else np.array(adata.X)
        self.expr     = torch.tensor(X,                                 dtype=torch.float32)
        self.tf       = torch.tensor(adata.obsm["X_tf"],                dtype=torch.float32)
        self.pathway  = torch.tensor(adata.obsm["X_pathway"],           dtype=torch.float32)
        self.go       = torch.tensor(adata.obsm["X_go"],                dtype=torch.float32)
        self.lineage  = torch.tensor(adata.obs["lineage_code"].values,  dtype=torch.long)
        self.celltype = torch.tensor(adata.obs["celltype_code"].values, dtype=torch.long)
        self.n_genes     = self.expr.shape[1]
        self.n_tfs       = self.tf.shape[1]
        self.n_pathways  = self.pathway.shape[1]
        self.n_go        = self.go.shape[1]
        self.n_lineages  = int(self.lineage.max().item()) + 1
        self.n_celltypes = int(self.celltype.max().item()) + 1
    def __len__(self): return self.expr.shape[0]
    def __getitem__(self, idx):
        return {"expr": self.expr[idx], "tf": self.tf[idx],
                "pathway": self.pathway[idx], "go": self.go[idx],
                "lineage": self.lineage[idx], "celltype": self.celltype[idx],
                "cell_idx": idx}

class BINNInferenceDataset(Dataset):
    def __init__(self, adata):
        X = adata.X.toarray() if issparse(adata.X) else np.array(adata.X)
        self.expr    = torch.tensor(X,                       dtype=torch.float32)
        self.tf      = torch.tensor(adata.obsm["X_tf"],      dtype=torch.float32)
        self.pathway = torch.tensor(adata.obsm["X_pathway"], dtype=torch.float32)
        self.go      = torch.tensor(adata.obsm["X_go"],      dtype=torch.float32)
    def __len__(self): return self.expr.shape[0]
    def __getitem__(self, idx):
        return {"expr": self.expr[idx], "tf": self.tf[idx],
                "pathway": self.pathway[idx], "go": self.go[idx],
                "cell_idx": idx}

dataset_train_full = BINNDataset(adata_train, barcoded_only=True)
n_val   = int(len(dataset_train_full) * 0.15)
n_train = len(dataset_train_full) - n_val
dataset_train, dataset_val = random_split(
    dataset_train_full, [n_train, n_val],
    generator=torch.Generator().manual_seed(42),
)
print(f"\n  Training set   : {n_train:,} cells")
print(f"  Validation set : {n_val:,} cells")

lineage_counts = torch.bincount(dataset_train_full.lineage)
class_weights  = 1.0 / lineage_counts.float()
class_weights  = class_weights / class_weights.sum() * len(lineage_counts)
class_weights  = class_weights.to(device)

# ══════════════════════════════════════════════════════════════════════════════
# Step 5 — Model architecture
# ══════════════════════════════════════════════════════════════════════════════
class GradientReversal(Function):
    @staticmethod
    def forward(ctx, x, alpha): ctx.alpha = alpha; return x.clone()
    @staticmethod
    def backward(ctx, g): return -ctx.alpha * g, None

def grad_reverse(x, alpha=1.0): return GradientReversal.apply(x, alpha)

class ModalityEncoder(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, out_dim), nn.LayerNorm(out_dim), nn.GELU(),
        )
    def forward(self, x): return self.net(x)

class ModalityEncoders(nn.Module):
    def __init__(self, dims, token_dim=TOKEN_DIM):
        super().__init__()
        self.enc_expr    = ModalityEncoder(dims["n_genes"],    512, token_dim, dropout=0.3)
        self.enc_tf      = ModalityEncoder(dims["n_tfs"],      128, token_dim, dropout=0.3)
        self.enc_pathway = ModalityEncoder(dims["n_pathways"],  32, token_dim, dropout=0.2)
        self.enc_go      = ModalityEncoder(dims["n_go"],       256, token_dim, dropout=0.3)
    def forward(self, expr, tf, pathway, go):
        return self.enc_expr(expr), self.enc_tf(tf), self.enc_pathway(pathway), self.enc_go(go)

class CrossModalFusion(nn.Module):
    def __init__(self, token_dim=TOKEN_DIM, n_heads=4, dropout=0.1):
        super().__init__()
        self.attn  = nn.MultiheadAttention(embed_dim=token_dim, num_heads=n_heads,
                                            dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(token_dim)
        self.ff    = nn.Sequential(nn.Linear(token_dim, token_dim*2), nn.GELU(),
                                    nn.Linear(token_dim*2, token_dim))
        self.norm2 = nn.LayerNorm(token_dim)
        self.last_attn_weights = None
    def forward(self, tokens):
        attn_out, attn_weights = self.attn(tokens, tokens, tokens,
                                            need_weights=True, average_attn_weights=True)
        self.last_attn_weights = attn_weights.detach()
        tokens = self.norm1(tokens + attn_out)
        tokens = self.norm2(tokens + self.ff(tokens))
        return tokens.mean(dim=1)

class LatentSpace(nn.Module):
    def __init__(self, in_dim, z_lineage=Z_LINEAGE, z_celltype=Z_CELLTYPE):
        super().__init__()
        self.fc_lineage  = nn.Sequential(nn.Linear(in_dim, z_lineage),
                                          nn.LayerNorm(z_lineage), nn.GELU())
        self.fc_celltype = nn.Sequential(nn.Linear(in_dim, z_celltype),
                                          nn.LayerNorm(z_celltype), nn.GELU())
    def forward(self, x): return self.fc_lineage(x), self.fc_celltype(x)

class LineageHead(nn.Module):
    def __init__(self, z_dim, n_lineages):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(z_dim, 128), nn.LayerNorm(128), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(128, 64), nn.GELU(), nn.Dropout(0.1), nn.Linear(64, n_lineages),
        )
    def forward(self, z): return self.net(z)

class CelltypeHead(nn.Module):
    def __init__(self, z_dim, n_celltypes):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(z_dim, 32), nn.GELU(),
                                  nn.Linear(32, n_celltypes))
    def forward(self, z): return self.net(z)

class ModalityAuxHead(nn.Module):
    def __init__(self, token_dim, n_lineages):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(token_dim, 32), nn.GELU(),
                                  nn.Linear(32, n_lineages))
    def forward(self, z): return self.net(z)

class BINN(nn.Module):
    def __init__(self, dims, token_dim=TOKEN_DIM,
                 z_lineage=Z_LINEAGE, z_celltype=Z_CELLTYPE):
        super().__init__()
        self.encoders      = ModalityEncoders(dims, token_dim=token_dim)
        self.fusion        = CrossModalFusion(token_dim=token_dim, n_heads=4)
        self.latent        = LatentSpace(in_dim=token_dim,
                                         z_lineage=z_lineage, z_celltype=z_celltype)
        self.lineage_head  = LineageHead(z_lineage, dims["n_lineages"])
        self.celltype_head = CelltypeHead(z_lineage, dims["n_celltypes"])
        self.aux_expr      = ModalityAuxHead(token_dim, dims["n_lineages"])
        self.aux_tf        = ModalityAuxHead(token_dim, dims["n_lineages"])
        self.aux_pathway   = ModalityAuxHead(token_dim, dims["n_lineages"])
        self.aux_go        = ModalityAuxHead(token_dim, dims["n_lineages"])
    def forward(self, batch, alpha=0.0):
        z_e, z_t, z_p, z_g = self.encoders(
            batch["expr"], batch["tf"], batch["pathway"], batch["go"])
        aux_logits = {"expr": self.aux_expr(z_e), "tf": self.aux_tf(z_t),
                      "pathway": self.aux_pathway(z_p), "go": self.aux_go(z_g)}
        tokens  = torch.stack([z_e, z_t, z_p, z_g], dim=1)
        z_fused = self.fusion(tokens)
        z_lin, z_ct = self.latent(z_fused)
        return {"logits_lineage":  self.lineage_head(z_lin),
                "logits_celltype": self.celltype_head(grad_reverse(z_lin, alpha)),
                "aux_logits": aux_logits, "z_lineage": z_lin, "z_celltype": z_ct,
                "attn_weights": self.fusion.last_attn_weights}

class BINNLoss(nn.Module):
    def __init__(self, class_weights=None):
        super().__init__()
        self.ce   = nn.CrossEntropyLoss(weight=class_weights)
        self.ce_u = nn.CrossEntropyLoss()
    def forward(self, out, batch):
        l_lin = self.ce(out["logits_lineage"],    batch["lineage"])
        l_adv = self.ce_u(out["logits_celltype"], batch["celltype"])
        l_aux = sum(self.ce(v, batch["lineage"])
                    for v in out["aux_logits"].values()) / 4
        return 1.0*l_lin + 0.1*l_adv + 0.3*l_aux, \
               {"lineage": l_lin.item(), "adv": l_adv.item(), "aux": l_aux.item()}

# ══════════════════════════════════════════════════════════════════════════════
# Step 6 — Train on single sample
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n── Step 6: Train on {TRAIN_SAMPLE} ──────────────────────────────────")

model     = BINN(dims).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS)
criterion = BINNLoss(class_weights=class_weights)
loader_tr = DataLoader(dataset_train, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
loader_va = DataLoader(dataset_val,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

best_bacc  = 0.0
best_state = None
no_imp     = 0

for epoch in range(1, N_EPOCHS + 1):
    progress = max(0, (epoch - 40) / (N_EPOCHS - 40))
    alpha    = float(2 / (1 + np.exp(-10 * progress)) - 1) if epoch >= 40 else 0.0

    model.train()
    for batch in loader_tr:
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()
        loss, _ = criterion(model(batch, alpha), batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for batch in loader_va:
            batch = {k: v.to(device) for k, v in batch.items()}
            preds.extend(model(batch)["logits_lineage"].argmax(1).cpu().numpy())
            labels.extend(batch["lineage"].cpu().numpy())
    scheduler.step()
    bacc = balanced_accuracy_score(labels, preds) * 100

    if bacc > best_bacc:
        best_bacc  = bacc
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        no_imp = 0; star = "★"
    else:
        no_imp += 1; star = ""

    if epoch % 10 == 0 or epoch == 1:
        print(f"  Epoch {epoch:3d} α={alpha:.2f}  val_bacc={bacc:.1f}%  {star}")

    if epoch >= MIN_EPOCHS and no_imp >= PATIENCE:
        print(f"\n  Early stopping at epoch {epoch} — best bacc={best_bacc:.1f}%")
        break

model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
print(f"\n  Best val bacc: {best_bacc:.1f}%")

# ══════════════════════════════════════════════════════════════════════════════
# Step 7 — Temperature calibration
# ══════════════════════════════════════════════════════════════════════════════
class TemperatureScaler(nn.Module):
    def __init__(self): super().__init__(); self.temperature = nn.Parameter(torch.ones(1))
    def forward(self, logits): return logits / self.temperature

scaler  = TemperatureScaler().to(device)
optim_T = torch.optim.LBFGS([scaler.temperature], lr=0.01, max_iter=100)
nll     = nn.CrossEntropyLoss()
all_logits, all_labels_cal = [], []
model.eval()
with torch.no_grad():
    for batch in loader_va:
        batch = {k: v.to(device) for k, v in batch.items()}
        all_logits.append(model(batch)["logits_lineage"].cpu())
        all_labels_cal.append(batch["lineage"].cpu())
all_logits     = torch.cat(all_logits)
all_labels_cal = torch.cat(all_labels_cal)

def eval_T():
    optim_T.zero_grad()
    nll(all_logits / scaler.temperature, all_labels_cal).backward()
    return nll(all_logits / scaler.temperature, all_labels_cal)
optim_T.step(eval_T)
print(f"  Temperature T = {scaler.temperature.item():.4f}")

# ══════════════════════════════════════════════════════════════════════════════
# Step 8 — Inference on all cells
# ══════════════════════════════════════════════════════════════════════════════
print("\n── Step 8: Inference on all cells ───────────────────────────────────")

def run_inference(adata_sub, model, scaler, le_lineage, device, batch_size=512):
    dataset = BINNInferenceDataset(adata_sub)
    loader  = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    model.eval(); scaler.eval()
    all_z, all_probs, all_preds = [], [], []
    with torch.no_grad():
        for batch in loader:
            batch_gpu = {k: v.to(device) for k, v in batch.items() if k != "cell_idx"}
            out       = model(batch_gpu)
            probs     = torch.softmax(scaler(out["logits_lineage"]), dim=-1)
            all_z.append(out["z_lineage"].cpu().numpy())
            all_probs.append(probs.cpu().numpy())
            all_preds.append(probs.argmax(1).cpu().numpy())
    probs_arr = np.concatenate(all_probs)
    preds     = np.concatenate(all_preds)
    return le_lineage.inverse_transform(preds), probs_arr.max(1), np.concatenate(all_z)

pred_lin, pred_conf, pred_z = run_inference(adata, model, scaler, le_lineage, device)
adata.obs["pseudo_lineage_ss"]      = pred_lin
adata.obs["pseudo_lineage_conf_ss"] = pred_conf
adata.obsm["X_lineage_ss"]          = pred_z
adata.obs["sample_type"] = adata.obs[SAMPLE_COL].apply(
    lambda s: "NC" if s in NC_SAMPLES else
              ("train" if s == TRAIN_SAMPLE else "held-out"))

print(f"  Inference complete on {adata.n_obs:,} cells")

# ══════════════════════════════════════════════════════════════════════════════
# Step 9 — Per-sample evaluation using 'sample' column
# ══════════════════════════════════════════════════════════════════════════════
print("\n── Step 9: Per-sample evaluation ────────────────────────────────────")
print(f"{'Sample':14s}  {'Type':10s}  {'N':>7s}  {'N_bc':>6s}  "
      f"{'Bacc':>7s}  {'ARI':>7s}  {'NMI':>7s}  {'MeanConf':>9s}")
print("─" * 80)

results = []
for s in sorted(adata.obs[SAMPLE_COL].unique()):
    s_mask  = adata.obs[SAMPLE_COL] == s
    s_data  = adata[s_mask]
    bc_mask = s_data.obs["lineage_code"] >= 0
    stype   = "NC" if s in NC_SAMPLES else \
              ("TRAIN" if s == TRAIN_SAMPLE else "held-out")
    n_bc   = bc_mask.sum()
    conf   = s_data.obs["pseudo_lineage_conf_ss"].mean()

    if n_bc >= 5:
        y_true = s_data[bc_mask].obs["lineage_code"].values
        y_pred = le_lineage.transform(s_data[bc_mask].obs["pseudo_lineage_ss"].values)
        bacc   = balanced_accuracy_score(y_true, y_pred) * 100
        ari    = adjusted_rand_score(y_true, y_pred)
        nmi    = normalized_mutual_info_score(y_true, y_pred)
    else:
        bacc = ari = nmi = float("nan")

    results.append({"sample": s, "type": stype, "n_total": s_mask.sum(),
                    "n_barcoded": n_bc, "bacc": bacc, "ari": ari,
                    "nmi": nmi, "mean_conf": conf})
    print(f"  {s:14s}  {stype:10s}  {s_mask.sum():7,d}  {n_bc:6,d}  "
          f"{bacc:6.1f}%  {ari:7.3f}  {nmi:7.3f}  {conf:8.3f}")

results_df = pd.DataFrame(results)

# Summary
tr        = results_df[results_df["sample"] == TRAIN_SAMPLE].iloc[0]
inj_ho    = results_df[(results_df["type"] == "held-out") & results_df["ari"].notna()]
nc_res    = results_df[results_df["type"] == "NC"]
ho_conf   = results_df[results_df["type"] == "held-out"]["mean_conf"].mean()
nc_conf   = nc_res["mean_conf"].mean()

print(f"\n── Summary ───────────────────────────────────────────────────────────")
print(f"  Train ({TRAIN_SAMPLE}):  bacc={tr['bacc']:.1f}%  ARI={tr['ari']:.3f}")
if len(inj_ho):
    print(f"  Held-out injected:  mean bacc={inj_ho['bacc'].mean():.1f}%  "
          f"ARI={inj_ho['ari'].mean():.3f}  NMI={inj_ho['nmi'].mean():.3f}")
print(f"  NC controls (no ground truth):")
print(f"    mean conf = {nc_conf:.3f}  vs  held-out conf = {ho_conf:.3f}")
diff = abs(nc_conf - ho_conf)
print(f"    difference = {diff:.3f}  — "
      f"{'similar confidence (NC cells map to known lineages)' if diff < 0.05 else 'lower confidence (NC cells are transcriptionally distinct)'}")

# ══════════════════════════════════════════════════════════════════════════════
# Step 10 — Visualisation: 3 columns per sample (true / predicted / confidence)
#            Grouped: train first, then held-out injected, then NC
# ══════════════════════════════════════════════════════════════════════════════
print("\n── Step 10: Visualisation ───────────────────────────────────────────")

sample_order = (
    [TRAIN_SAMPLE] +
    sorted([s for s in adata.obs[SAMPLE_COL].unique()
            if s != TRAIN_SAMPLE and s not in NC_SAMPLES]) +
    sorted(NC_SAMPLES)
)
# Only include samples present in adata
sample_order = [s for s in sample_order if s in adata.obs[SAMPLE_COL].values]

n_rows = len(sample_order)
fig, axes = plt.subplots(n_rows, 3, figsize=(18, n_rows * 4.5), facecolor="white")
if n_rows == 1:
    axes = axes[np.newaxis, :]

for ri, s in enumerate(sample_order):
    s_mask  = adata.obs[SAMPLE_COL] == s
    s_adata = adata[s_mask]
    bc_mask = s_adata.obs["lineage_code"] >= 0
    stype   = "NC" if s in NC_SAMPLES else \
              ("TRAIN" if s == TRAIN_SAMPLE else "held-out")

    # Panel 1 — true c2v lineage (barcoded cells only)
    ax1 = axes[ri, 0]
    sc.pl.embedding(s_adata, basis=UMAP_KEY,
                    color="c2v_leiden",
                    title=f"{s}  [{stype}]\ntrue c2v lineage (barcoded only)",
                    frameon=False, legend_loc="on data", legend_fontsize=5,
                    ax=ax1, show=False, size=8)

    # Panel 2 — predicted pseudo-lineage (all cells)
    ax2 = axes[ri, 1]
    sc.pl.embedding(s_adata, basis=UMAP_KEY,
                    color="pseudo_lineage_ss",
                    title="predicted pseudo-lineage (all cells)",
                    frameon=False, legend_loc="on data", legend_fontsize=5,
                    ax=ax2, show=False, size=8)

    # Panel 3 — confidence
    ax3 = axes[ri, 2]
    mean_c = s_adata.obs["pseudo_lineage_conf_ss"].mean()
    sc.pl.embedding(s_adata, basis=UMAP_KEY,
                    color="pseudo_lineage_conf_ss",
                    title=f"confidence  (mean = {mean_c:.3f})",
                    frameon=False, cmap="viridis", vmin=0, vmax=1,
                    ax=ax3, show=False, size=8)

    # ARI/bacc annotation
    row = results_df[results_df["sample"] == s].iloc[0]
    if not np.isnan(row["ari"]):
        ann = f"ARI={row['ari']:.3f}  bacc={row['bacc']:.1f}%"
    else:
        ann = f"no ground truth  conf={row['mean_conf']:.3f}"
    for ax in [ax1, ax2, ax3]:
        ax.text(0.02, 0.02, ann, transform=ax.transAxes, fontsize=8,
                color="#534AB7", fontweight="bold",
                bbox=dict(boxstyle="round,pad=0.2", facecolor="white",
                          alpha=0.8, edgecolor="#EEEDFE"))

if len(inj_ho):
    title_str = (f"Single-sample generalisation  ·  trained on: {TRAIN_SAMPLE}\n"
                 f"Held-out ARI={inj_ho['ari'].mean():.3f}  "
                 f"bacc={inj_ho['bacc'].mean():.1f}%  ·  "
                 f"NC conf={nc_conf:.3f}  held-out conf={ho_conf:.3f}")
else:
    title_str = f"Single-sample generalisation  ·  trained on: {TRAIN_SAMPLE}"

plt.suptitle(title_str, fontsize=12, fontweight="bold", y=1.002)
plt.tight_layout()
out_path = os.path.join(SAVE_DIR, "single_sample_generalisation.png")
plt.savefig(out_path, dpi=150, bbox_inches="tight", facecolor="white")
plt.show()
print(f"Saved: {out_path}")

# ══════════════════════════════════════════════════════════════════════════════
# Step 11 — Save
# ══════════════════════════════════════════════════════════════════════════════
torch.save({
    "model_state_dict":  model.state_dict(),
    "scaler_state_dict": scaler.state_dict(),
    "dims":              dims,
    "train_sample":      TRAIN_SAMPLE,
    "val_bacc":          best_bacc,
}, os.path.join(SAVE_DIR, "binn_single_sample_model.pt"))

results_df.to_csv(os.path.join(SAVE_DIR, "single_sample_results.csv"), index=False)
print(f"Saved: binn_single_sample_model.pt")
print(f"Saved: single_sample_results.csv")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Step 9b — Deeper analysis of results
# ══════════════════════════════════════════════════════════════════════════════

# ── 1. NC barcoded cells are contamination ────────────────────────────────────
print("\n── NC barcode contamination check ───────────────────────────────────")
for nc_s in NC_SAMPLES:
    nc_mask_s = adata.obs[SAMPLE_COL] == nc_s
    nc_bc     = adata.obs.loc[nc_mask_s & (adata.obs["lineage_code"] >= 0), "cloneid"]
    inj_bc    = adata.obs.loc[(~adata.obs[SAMPLE_COL].isin(NC_SAMPLES)) &
                               (adata.obs["lineage_code"] >= 0), "cloneid"]
    overlap   = set(nc_bc.values) & set(inj_bc.values)
    print(f"  {nc_s}: {len(nc_bc)} 'barcoded' cells  —  "
          f"{len(overlap)} clone IDs shared with injected samples  "
          f"({'contamination' if len(overlap) > 0 else 'unique — investigate'})")

# ── 2. Per-sample held-out ARI vs number of barcoded cells ───────────────────
print("\n── Held-out performance vs sample size ──────────────────────────────")
inj_ho_df = results_df[(results_df["type"] == "held-out") &
                        results_df["ari"].notna() &
                        (results_df["n_barcoded"] >= 10)].copy()

# Is there a relationship between n_barcoded and ARI?
corr = inj_ho_df[["n_barcoded", "ari"]].corr().iloc[0, 1]
print(f"  Correlation between n_barcoded and ARI: {corr:.3f}")
print(f"  Samples with ARI > 0.4:  {(inj_ho_df['ari'] > 0.4).sum()}")
print(f"  Samples with ARI < 0.4:  {(inj_ho_df['ari'] < 0.4).sum()}")

# ── 3. Lineage distribution: are NC cells assigned to same lineages? ──────────
print("\n── Lineage assignment distribution ──────────────────────────────────")
train_dist = (adata.obs.loc[adata.obs[SAMPLE_COL] == TRAIN_SAMPLE,
                             "pseudo_lineage_ss"]
              .value_counts(normalize=True).sort_index())
nc_dist    = (adata.obs.loc[adata.obs[SAMPLE_COL].isin(NC_SAMPLES),
                             "pseudo_lineage_ss"]
              .value_counts(normalize=True).sort_index())
ho_dist    = (adata.obs.loc[(results_df.set_index("sample")
                              .reindex(adata.obs[SAMPLE_COL])["type"] == "held-out").values,
                             "pseudo_lineage_ss"]
              .value_counts(normalize=True).sort_index())

from scipy.spatial.distance import jensenshannon
dist_df = pd.DataFrame({
    "train":    train_dist,
    "held_out": ho_dist,
    "NC":       nc_dist,
}).fillna(0)

jsd_train_ho = jensenshannon(dist_df["train"], dist_df["held_out"])
jsd_train_nc = jensenshannon(dist_df["train"], dist_df["NC"])
jsd_ho_nc    = jensenshannon(dist_df["held_out"], dist_df["NC"])

print(f"  JSD train vs held-out : {jsd_train_ho:.4f}")
print(f"  JSD train vs NC       : {jsd_train_nc:.4f}")
print(f"  JSD held-out vs NC    : {jsd_ho_nc:.4f}")
print(f"\n  Interpretation:")
if jsd_ho_nc < 0.15:
    print(f"  NC lineage distribution closely matches held-out injected cells.")
    print(f"  NC cells are being assigned to the same lineages as injected cells,")
    print(f"  suggesting shared transcriptional trajectories despite no barcode.")
else:
    print(f"  NC lineage distribution differs from held-out injected cells.")
    print(f"  NC cells may occupy different transcriptional states.")

# ── 4. Confidence by cell type within NC ─────────────────────────────────────
print("\n── NC confidence by cell type ───────────────────────────────────────")
nc_data = adata[adata.obs[SAMPLE_COL].isin(NC_SAMPLES)].copy()
if "majority_voting" in nc_data.obs.columns:
    nc_ct_conf = (nc_data.obs.groupby("majority_voting")["pseudo_lineage_conf_ss"]
                  .agg(["mean", "count"])
                  .sort_values("mean", ascending=False))
    nc_ct_conf.columns = ["mean_conf", "n_cells"]
    print(nc_ct_conf.head(10).to_string())
    print(f"\n  High-confidence NC cell types may represent lineages")
    print(f"  with conserved transcriptional signatures across animals.")

# ── 5. Quick scatter: n_barcoded vs ARI per held-out sample ──────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5), facecolor="white")

# Panel 1: n_barcoded vs ARI
ax = axes[0]
ax.scatter(inj_ho_df["n_barcoded"], inj_ho_df["ari"],
           color="#534AB7", s=80, alpha=0.8, edgecolors="white", lw=0.5)
for _, r in inj_ho_df.iterrows():
    ax.annotate(r["sample"], (r["n_barcoded"], r["ari"]),
                textcoords="offset points", xytext=(4, 3), fontsize=7, color="#888")
ax.set_xlabel("N barcoded cells in sample")
ax.set_ylabel("ARI vs clone2vec ground truth")
ax.set_title("Held-out ARI vs sample size")
ax.axhline(y=inj_ho_df["ari"].mean(), color="#888", lw=0.8, linestyle="--",
           label=f"mean ARI={inj_ho_df['ari'].mean():.3f}")
ax.legend(fontsize=8)
ax.spines[["top", "right"]].set_visible(False)

# Panel 2: confidence distribution by group
ax = axes[1]
groups = {"Train": adata.obs.loc[adata.obs[SAMPLE_COL]==TRAIN_SAMPLE,
                                  "pseudo_lineage_conf_ss"],
          "Held-out": adata.obs.loc[adata.obs["sample_type"]=="held-out",
                                     "pseudo_lineage_conf_ss"],
          "NC":       adata.obs.loc[adata.obs["sample_type"]=="NC",
                                     "pseudo_lineage_conf_ss"]}
colors = {"Train": "#534AB7", "Held-out": "#1D9E75", "NC": "#D85A30"}
for label, conf in groups.items():
    ax.hist(conf, bins=40, alpha=0.55, label=label,
            color=colors[label], density=True)
ax.set_xlabel("Prediction confidence")
ax.set_ylabel("Density")
ax.set_title("Confidence distribution by group")
ax.legend(fontsize=9)
ax.spines[["top", "right"]].set_visible(False)

# Panel 3: lineage distribution comparison
ax = axes[2]
lin_labels = sorted(dist_df.index, key=lambda v: int(v) if v.isdigit() else v)
x = np.arange(len(lin_labels))
w = 0.28
ax.bar(x - w,   [dist_df.loc[l, "train"]    for l in lin_labels],
       width=w, color="#534AB7", alpha=0.8, label="Train")
ax.bar(x,       [dist_df.loc[l, "held_out"] for l in lin_labels],
       width=w, color="#1D9E75", alpha=0.8, label="Held-out")
ax.bar(x + w,   [dist_df.loc[l, "NC"]       for l in lin_labels],
       width=w, color="#D85A30", alpha=0.8, label="NC")
ax.set_xticks(x)
ax.set_xticklabels([f"L{l}" for l in lin_labels], rotation=90, fontsize=7)
ax.set_ylabel("Fraction of cells assigned")
ax.set_title(f"Lineage distribution by group\nJSD(held-out,NC)={jsd_ho_nc:.3f}")
ax.legend(fontsize=8)
ax.spines[["top", "right"]].set_visible(False)

plt.suptitle(
    f"Single-sample generalisation analysis  ·  trained on {TRAIN_SAMPLE}\n"
    f"Held-out mean ARI={inj_ho_df['ari'].mean():.3f}  ·  "
    f"NC conf={nc_dist.mean():.3f}  ·  Barcode corr r={corr:.3f}",
    fontsize=11, fontweight="bold"
)
plt.tight_layout()
out_path = os.path.join(SAVE_DIR, "single_sample_analysis.png")
plt.savefig(out_path, dpi=150, bbox_inches="tight", facecolor="white")
plt.show()
print(f"Saved: {out_path}")